In [2]:
import os, time, math, sys, json, re, gc, warnings
from typing import Dict, Tuple, Optional

import numpy as np
import pandas as pd
import polars as pl
import h5py

import scanpy as sc

# Optional (used later in the notebook; harmless to import here)
# import squidpy as sq
# from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, f1_score, confusion_matrix
# import umap

warnings.filterwarnings("ignore", category=UserWarning)
pl.Config.set_fmt_str_lengths(120)


tmp_dir= "/homes/olymp/julius.guntrum/tmp"
OUT_DIR = os.path.join(tmp_dir,"Koeln/skin_car")

In [ ]:
sksksdlkjdhd dikjdhdkjbd luhsuhsiuskuhs liushsluh   ukkkhjhjhssliugs sdvssssdsvdgrrrrrgeddrgdrgdfffdrgdfdrgddf

In [2]:
SHARD_GLOB = os.path.join(OUT_DIR, "family_geneaware_shard_*.parquet")
MANIFEST = os.path.join(OUT_DIR, "family_geneaware_shards.txt")
REF_ADATA = os.path.join(tmp_dir,"Koeln/bone/.h5ad")

In [6]:
# ============================================
# Global config: imports, paths, parameters
# ============================================
import os, time, math, sys, json, re, gc, warnings
from typing import Dict, Tuple, Optional

import numpy as np
import pandas as pd
import polars as pl
import h5py

import scanpy as sc

# Optional (used later in the notebook; harmless to import here)
# import squidpy as sq
# from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, f1_score, confusion_matrix
# import umap

warnings.filterwarnings("ignore", category=UserWarning)
pl.Config.set_fmt_str_lengths(120)


tmp_dir= "/homes/olymp/julius.guntrum/tmp"
# ---------- User paths ----------
# Space Ranger v4+ output (single slide, deep sequencing run)
OUTS_DIR = "/mnt/ribolution/secondary_analysis/2024_SpatSeq_VisiumHD/240726_VH00421_54_2223G3WNX/results-rerun/koeln/C24_13883/outs/"
MOLECULE_INFO = os.path.join(OUTS_DIR, "molecule_info.h5")                   # required
MAPPING_PARQUET = os.path.join(OUTS_DIR, "barcode_mappings.parquet")         # required for 2µm barcode -> segmented cell

# Reference AnnData (deep run; segmented cell x gene)
#REF_ADATA = os.path.join(tmp_dir,"Koeln/skin_car/skin_car_ref.h5ad")

# Workspace for intermediate shards & outputs
OUT_DIR = os.path.join(tmp_dir,"Koeln/bone")
os.makedirs(OUT_DIR, exist_ok=True)

# Output shard manifest
SHARD_GLOB = os.path.join(OUT_DIR, "family_geneaware_shard_*.parquet")
MANIFEST = os.path.join(OUT_DIR, "family_geneaware_shards.txt")

In [12]:
OUT_DIR

'/homes/olymp/julius.guntrum/tmp/Koeln/bone'

In [17]:
# ============================================
# Global config: imports, paths, parameters
# ============================================
import os, time, math, sys, json, re, gc, warnings
from typing import Dict, Tuple, Optional

import numpy as np
import pandas as pd
import polars as pl
import h5py

import scanpy as sc

# Optional (used later in the notebook; harmless to import here)
# import squidpy as sq
# from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, f1_score, confusion_matrix
# import umap

warnings.filterwarnings("ignore", category=UserWarning)
pl.Config.set_fmt_str_lengths(120)


tmp_dir= "/homes/olymp/julius.guntrum/tmp"
# ---------- User paths ----------
# Space Ranger v4+ output (single slide, deep sequencing run)
OUTS_DIR = "/mnt/ribolution/secondary_analysis/2024_SpatSeq_VisiumHD/240726_VH00421_54_2223G3WNX/results-rerun/koeln/24_13076/outs"
MOLECULE_INFO = os.path.join(OUTS_DIR, "molecule_info.h5")                   # required
MAPPING_PARQUET = os.path.join(OUTS_DIR, "barcode_mappings.parquet")         # required for 2µm barcode -> segmented cell

# Reference AnnData (deep run; segmented cell x gene)
REF_ADATA = os.path.join(tmp_dir,"Koeln/duo/subsamp/duo_full_ref.h5ad")

# Workspace for intermediate shards & outputs
OUT_DIR = os.path.join(tmp_dir,"Koeln/duo/subsamp")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- General parameters ----------
RNG_SEED = 1                            # global RNG seed for later steps (not used in this chunk)
N_THREADS = int(os.environ.get("OMP_NUM_THREADS", "8"))

# molecule_info streaming / shard writing
CHUNK_MOLECULES = 20_000_000            # tune based on RAM; >=5M usually safe
KEEP_FEATURE_TYPE_SUBSTR = "gene expression"  # filter features/rows to this type (case-insensitive)

# Mapping parquet columns (Space Ranger v4+ VisiumHD cell segmentation)
CB_COL = "square_002um"                 # 2µm barcode / square id
CELLID_COL = "cell_id"                  # segmented cell id
IN_CELL_COL = "in_cell"                 # boolean; 2µm square lies inside segmented cell
STRIP_CB_SUFFIX = True                  # strip "-1" from barcodes when matching

# Output shard manifest
SHARD_GLOB = os.path.join(OUT_DIR, "family_geneaware_shard_*.parquet")
MANIFEST = os.path.join(OUT_DIR, "family_geneaware_shards.txt")

# ---------- Sanity checks ----------
assert os.path.exists(MOLECULE_INFO), f"Missing file: {MOLECULE_INFO}"
assert os.path.exists(MAPPING_PARQUET), f"Missing file: {MAPPING_PARQUET}"
assert os.path.exists(REF_ADATA), f"Missing file: {REF_ADATA}"

# Load the deep reference AnnData (segmented cells)
adata = sc.read_h5ad(REF_ADATA)
assert adata.n_obs > 0 and adata.n_vars > 0, "Empty AnnData loaded."

In [18]:

 
# ============================================
# Task 1: Build per-UMI, gene-aware family table from molecule_info.h5
# ============================================

def _strip_dash1(s: str) -> str:
    s = "" if s is None else str(s)
    return s[:-2] if s.endswith("-1") else s

def _strip_version(x: str) -> str:
    """Remove Ensembl version suffix, e.g., ENSG000001.13 -> ENSG000001."""
    x = "" if x is None else str(x)
    return x.split(".")[0] if "." in x else x

def build_gene_indexers(adata) -> Tuple[Dict[str, int], Dict[str, int]]:
    """
    Return two dicts:
      - id_to_idx: versionless stable ID (Ensembl-like) -> var index
      - sym_to_idx: gene symbol/name -> var index
    Priority will be ID first, then symbol as fallback.
    """
    var = adata.var.copy()
    # Try to detect common gene id / symbol columns
    candidate_id_cols = [c for c in ["gene_ids", "gene_id", "feature_id", "ensembl_id"] if c in var.columns]
    candidate_sym_cols = [c for c in ["gene_symbols", "gene_symbol", "gene_name", "symbol"] if c in var.columns]

    # Build ID mapper (versionless)
    id_to_idx = {}
    if candidate_id_cols:
        id_col = candidate_id_cols[0]
        ids = var[id_col].astype(str).map(_strip_version)
        id_to_idx = {ids.iat[i]: i for i in range(len(ids))}
    else:
        # If var_names are IDs (common in Ensembl-based matrices)
        vn = pd.Index(adata.var_names.astype(str).map(_strip_version))
        # Only keep entries that look like Ensembl (start with ENS)
        id_to_idx = {vn[i]: i for i in range(len(vn)) if vn[i].startswith("ENS")}

    # Build symbol mapper (fallback)
    sym_to_idx = {}
    if candidate_sym_cols:
        sym_col = candidate_sym_cols[0]
        syms = var[sym_col].astype(str)
        sym_to_idx = {syms.iat[i]: i for i in range(len(syms))}
    else:
        # If var_names are symbols (typical when IDs are not present)
        vn = pd.Index(adata.var_names.astype(str))
        sym_to_idx = {vn[i]: i for i in range(len(vn))}

    return id_to_idx, sym_to_idx

# ----------------------------
# 1) Build 2µm barcode -> segmented cell_idx map (only in-cell + non-empty cell_id)
# ----------------------------
t0 = time.time()
print("[map] loading barcode_mappings.parquet ...")
lf = pl.scan_parquet(MAPPING_PARQUET)
cols = set(lf.collect_schema().names())
for c in [CB_COL, CELLID_COL, IN_CELL_COL]:
    assert c in cols, f"Column '{c}' missing in mapping parquet."

lf2 = (
    lf.filter(pl.col(IN_CELL_COL) == True)
      .filter(pl.col(CELLID_COL).is_not_null() & (pl.col(CELLID_COL) != ""))
)

if STRIP_CB_SUFFIX:
    lf2 = lf2.with_columns(
        pl.col(CB_COL).cast(pl.Utf8).map_elements(_strip_dash1, return_dtype=pl.Utf8).alias("CB_norm")
    )
    CB_KEY = "CB_norm"
else:
    lf2 = lf2.with_columns(pl.col(CB_COL).cast(pl.Utf8).alias("CB_norm"))
    CB_KEY = CB_COL

# Map cell_id -> adata index
cellid_to_idx = {str(cid): i for i, cid in enumerate(adata.obs_names.astype(str))}
mp = lf2.select([CB_KEY, CELLID_COL]).collect().to_pandas()
mp[CELLID_COL] = mp[CELLID_COL].astype(str)
mp = mp[mp[CELLID_COL].isin(cellid_to_idx.keys())]

cb_to_cellidx: Dict[str, int] = {}
for cb, cid in zip(mp[CB_KEY].tolist(), mp[CELLID_COL].tolist()):
    if cb not in cb_to_cellidx:
        cb_to_cellidx[cb] = cellid_to_idx[cid]

print(f"[map] mapped CB→cell: {len(cb_to_cellidx):,} unique 2µm barcodes in {time.time()-t0:.1f}s")

# ----------------------------
# 2) Feature (gene) mapping: molecule_info features -> adata.var indices
# ----------------------------
print("[feat] loading features from molecule_info.h5 ...")
with h5py.File(MOLECULE_INFO, "r") as f:
    feat_id = f["features/id"][:]            # bytes (often Ensembl)
    feat_nm = f["features/name"][:]          # bytes (gene symbol)
    feat_tp = f["features/feature_type"][:]  # bytes

feat_id = np.array([x.decode("utf-8", "ignore") for x in feat_id], dtype=object)
feat_nm = np.array([x.decode("utf-8", "ignore") for x in feat_nm], dtype=object)
feat_tp = np.array([x.decode("utf-8", "ignore") for x in feat_tp], dtype=object)

u, c = np.unique(feat_tp, return_counts=True)
print("[feat] feature_type counts:")
for t, n in zip(u, c):
    print(f"       {t}: {n:,}")

keep_mask_type = np.array([KEEP_FEATURE_TYPE_SUBSTR.lower() in str(t).lower() for t in feat_tp], dtype=bool)
print(f"[feat] keeping types with '{KEEP_FEATURE_TYPE_SUBSTR}': {keep_mask_type.sum():,}/{feat_tp.size:,}")

# Build gene indexers from the reference AnnData
id_to_idx, sym_to_idx = build_gene_indexers(adata)

# Map features to var indices (prefer ID first, then symbol)
feat_to_gene_idx = np.full(feat_id.size, -1, dtype=np.int32)
hits_id = hits_sym = 0
for j in range(feat_id.size):
    if not keep_mask_type[j]:
        continue
    gi = -1
    fid = _strip_version(feat_id[j])
    if fid and fid in id_to_idx:
        gi = id_to_idx[fid]; hits_id += 1
    else:
        gnm = feat_nm[j]
        if gnm and gnm in sym_to_idx:
            gi = sym_to_idx[gnm]; hits_sym += 1
    feat_to_gene_idx[j] = gi

n_unmapped = int((feat_to_gene_idx < 0).sum())
print(f"[feat] mapped features → adata.var | ID={hits_id:,}, Symbol={hits_sym:,}, Unmapped={n_unmapped:,}")

# ----------------------------
# 3) Barcode index (molecule_info) -> adata cell_idx vector
# ----------------------------
print("[bar] loading molecule_info barcodes ...")
with h5py.File(MOLECULE_INFO, "r") as f:
    barcodes = f["barcodes"][:]  # bytes
barcodes = [x.decode("utf-8", "ignore") for x in barcodes]
if STRIP_CB_SUFFIX:
    barcodes = [_strip_dash1(s) for s in barcodes]

n_barcodes = len(barcodes)
barcode_to_cell_idx = np.full(n_barcodes, -1, dtype=np.int32)
hit = 0
for i, s in enumerate(barcodes):
    ci = cb_to_cellidx.get(s, -1)
    barcode_to_cell_idx[i] = ci
    if ci >= 0:
        hit += 1
print(f"[bar] barcodes mapped to a segmented cell: {hit:,} / {n_barcodes:,}")

# ----------------------------
# 4) Stream molecule_info rows -> per-UMI family shards (cell_idx, gene_idx, umi_reads, umi_id)
# ----------------------------
print("[fam] streaming molecules to per-UMI shards ...")
shards = []
with h5py.File(MOLECULE_INFO, "r") as f:
    n_mol = f["barcode_idx"].shape[0]
    n_chunks = math.ceil(n_mol / CHUNK_MOLECULES)
    print(f"[fam] molecules: {n_mol:,} | chunks: {n_chunks}")

    for k in range(n_chunks):
        start = k * CHUNK_MOLECULES
        end   = min((k + 1) * CHUNK_MOLECULES, n_mol)
        t1 = time.time()

        # Load chunk
        bc_idx  = f["barcode_idx"][start:end].astype(np.int64, copy=False)
        ft_idx  = f["feature_idx"][start:end].astype(np.int64, copy=False)
        umi     = f["umi"][start:end].astype(np.uint32, copy=False)
        reads   = f["count"][start:end].astype(np.int32, copy=False)

        # Map indices
        cells = barcode_to_cell_idx[bc_idx]
        genes = feat_to_gene_idx[ft_idx]

        # Keep only: in segmented cell, mapped gene, positive reads
        mask = (cells >= 0) & (genes >= 0) & (reads > 0)
        kept = int(mask.sum())

        if kept > 0:
            df_pl = pl.DataFrame({
                "cell_idx":  pl.Series(cells[mask].astype(np.int32)),
                "gene_idx":  pl.Series(genes[mask].astype(np.int32)),
                "umi_reads": pl.Series(reads[mask].astype(np.int32)),   # per-UMI-family read count (confident gene-mapped)
                "umi_id":    pl.Series(umi[mask].astype(np.uint32)),    # optional, useful for QA
            })
            shard = os.path.join(OUT_DIR, f"family_geneaware_shard_{k:04d}.parquet")
            df_pl.write_parquet(shard, compression="zstd")
            shards.append(shard)

        rate = (end - start) / max(1e-6, (time.time() - t1))
        print(f"[fam] chunk {k+1}/{n_chunks} rows {start:,}..{end-1:,} | kept={kept:,} | rate≈{rate:,.0f}/s")

if not shards:
    raise RuntimeError("No per-UMI families were collected; check mapping/filters.")

# Write a manifest so downstream can glob safely (and you have a record)
with open(MANIFEST, "w") as fh:
    for s in shards:
        fh.write(s + "\n")

# Quick QC via lazy scan (no full load)
lf_all = pl.scan_parquet(SHARD_GLOB)
qc = (
    lf_all.select([
        pl.len().alias("n_families"),
        pl.col("umi_reads").sum().alias("total_reads"),
        pl.col("cell_idx").n_unique().alias("n_cells_nonempty"),
        pl.col("gene_idx").n_unique().alias("n_genes_nonempty"),
        pl.col("umi_reads").mean().alias("mean_reads_per_family"),
    ])
    .collect()
    .to_pandas()
    .iloc[0]
    .to_dict()
)

fmt_qc = {k: (int(v) if isinstance(v, (np.integer, int)) else (round(float(v), 3))) for k, v in qc.items()}
print("[fam][QC]", fmt_qc)
print("[done] shards written. Manifest:", MANIFEST)


[map] loading barcode_mappings.parquet ...
[map] mapped CB→cell: 1,250,265 unique 2µm barcodes in 2.2s
[feat] loading features from molecule_info.h5 ...
[feat] feature_type counts:
       Gene Expression: 39,126
[feat] keeping types with 'gene expression': 39,126/39,126
[feat] mapped features → adata.var | ID=38,645, Symbol=0, Unmapped=481
[bar] loading molecule_info barcodes ...
[bar] barcodes mapped to a segmented cell: 1,250,265 / 11,222,500
[fam] streaming molecules to per-UMI shards ...
[fam] molecules: 45,921,859 | chunks: 3
[fam] chunk 1/3 rows 0..19,999,999 | kept=15,922,856 | rate≈6,045,530/s
[fam] chunk 2/3 rows 20,000,000..39,999,999 | kept=15,564,540 | rate≈6,073,412/s
[fam] chunk 3/3 rows 40,000,000..45,921,858 | kept=4,871,604 | rate≈6,045,690/s
[fam][QC] {'n_families': 36359000.0, 'total_reads': 156709756.0, 'n_cells_nonempty': 58240.0, 'n_genes_nonempty': 18337.0, 'mean_reads_per_family': 4.31}
[done] shards written. Manifest: /homes/olymp/julius.guntrum/tmp/Koeln/duo/s

In [2]:
# -------- CONFIG (re-use variables from previous chunk) --------
# Uses: OUT_DIR, REF_ADATA, MANIFEST pre-defined earlier
SAVE_DIR = os.path.join(OUT_DIR, "runs_rpc_clamped")
os.makedirs(SAVE_DIR, exist_ok=True)

In [7]:
# ============================================
# STEP 2b — Global, NESTED read-level subsampling with REPLICATES
#           RPC grid (feasible max), N_REPS per target
#           Outputs per-depth .h5ad with per-cell reads/UMIs/saturation
# ============================================
import os, time, json, hashlib, math
import numpy as np
import pandas as pd
import polars as pl
import scanpy as sc
from scipy import sparse
REF_ADATA = os.path.join(OUT_DIR, "/homes/olymp/julius.guntrum/tmp/Koeln/bone/bone_ref.h5ad")
# -------- CONFIG (re-use variables from previous chunk) --------
# Uses: OUT_DIR, REF_ADATA, MANIFEST pre-defined earlier
SAVE_DIR = os.path.join(OUT_DIR, "runs_rpc_clamped")
os.makedirs(SAVE_DIR, exist_ok=True)

LAYER_LOGNORM = "lognorm"

# Grid settings (depth levels; inclusive of max)
N_STEPS = 10        # equally spaced RPC targets (incl. max)
N_REPS  = 3         # number of replicate subsamples PER target RPC

# RNG seeds (replicate-level; nested across depths within a replicate)
RNG_SEED_BASE = 12345

# Micro-batching of shard rows (families) to limit peak RAM when vectorizing
FAMILY_BATCH_SIZE = 2_000_000

# ----------------------------------------

def _ensure_lognorm(ad, layer=LAYER_LOGNORM):
    if layer not in ad.layers:
        norm = sc.pp.normalize_total(ad, target_sum=1e4, inplace=False)
        ad.layers[layer] = norm["X"]
        sc.pp.log1p(ad, layer=layer)

def _read_shard(path):
    """Expect columns: cell_idx, gene_idx, umi_reads  (or legacy family_reads)."""
    try:
        df = pl.read_parquet(path, columns=["cell_idx","gene_idx","umi_reads"]).to_pandas()
    except Exception:
        df = pl.read_parquet(path, columns=["cell_idx","gene_idx","family_reads"]).to_pandas()
        df.rename(columns={"family_reads":"umi_reads"}, inplace=True)
    # dtypes
    df["cell_idx"]  = df["cell_idx"].astype(np.int32, copy=False)
    df["gene_idx"]  = df["gene_idx"].astype(np.int32, copy=False)
    df["umi_reads"] = df["umi_reads"].astype(np.int32, copy=False)
    return df

def _build_counts_matrix(n_cells, n_genes, grouped_triplets):
    if not grouped_triplets:
        return sparse.csr_matrix((n_cells, n_genes), dtype=np.int32)
    if len(grouped_triplets) == 1:
        rows, cols, dat = grouped_triplets[0]
    else:
        rows = np.concatenate([t[0] for t in grouped_triplets])
        cols = np.concatenate([t[1] for t in grouped_triplets])
        dat  = np.concatenate([t[2] for t in grouped_triplets])
    return sparse.coo_matrix((dat, (rows, cols)), shape=(n_cells, n_genes), dtype=np.int32).tocsr()

def _load_manifest(p):
    with open(p, "r") as f:
        paths = [ln.strip() for ln in f if ln.strip()]
    if not paths:
        raise RuntimeError(f"Empty manifest: {p}")
    return paths

def _rng_for_rep(rep:int) -> np.random.Generator:
    # stable per-replicate RNG; ensures nestedness across depths within a rep
    h = int(hashlib.blake2b(f"rep|{rep}".encode(), digest_size=8).hexdigest(), 16) & 0x7FFFFFFF
    return np.random.default_rng(RNG_SEED_BASE + h)

# ---------- Load reference + shards ----------
ad_ref = sc.read_h5ad(REF_ADATA)
n_cells, n_genes = ad_ref.n_obs, ad_ref.n_vars
print(f"[ref] {n_cells:,} cells × {n_genes:,} genes")

shards = _load_manifest(MANIFEST)
print(f"[manifest] {len(shards)} shards")

# ---------- Compute total reads present in shards (we thin these reads) ----------
reads_in_shards = 0
families_in_shards = 0
t0 = time.time()
for i, sh in enumerate(shards, 1):
    try:
        s = pl.read_parquet(sh, columns=["umi_reads"]).select(pl.col("umi_reads").sum()).item()
    except Exception:
        s = pl.read_parquet(sh, columns=["family_reads"]).select(pl.col("family_reads").sum()).item()
    reads_in_shards += int(s)

    fam_ct = pl.read_parquet(sh, columns=["cell_idx"]).height
    families_in_shards += int(fam_ct)
    if i == 1 or i % 2 == 0 or i == len(shards):
        print(f"  [scan] shard {i:02d}/{len(shards)} | +reads={int(s):,} | cum_reads={reads_in_shards:,}")
print(f"[totals] reads_in_shards={reads_in_shards:,} | families={families_in_shards:,} | {time.time()-t0:.1f}s")

# ---------- Determine feasible max RPC and build equally spaced target list ----------
max_rpc = int(np.floor(reads_in_shards / n_cells))
print(f"[feasible] max_rpc={max_rpc:,} (reads_in_shards {reads_in_shards:,} / cells {n_cells:,})")

if max_rpc <= 0:
    target_rpcs = []
else:
    # Equally spaced integers from 50 to max_rpc (inclusive), with N_STEPS points.
    target_rpcs = sorted(np.unique(np.linspace(50, max_rpc, N_STEPS, dtype=int)).tolist())
    target_rpcs = [t for t in target_rpcs if t > 0]

print("[targets]", target_rpcs)

# Precompute p_keep (in ascending order) and maintain a mapping to target_rpc
p_for_target = []
for t in target_rpcs:
    p = float(t) * float(n_cells) / float(reads_in_shards)
    p_for_target.append(float(np.clip(p, 0.0, 1.0)))

# Build unique, ascending p-grid for nested sampling
p_grid = np.array(sorted(set(p_for_target)))
K = len(p_grid)

# Helper: map a p_target to index in p_grid (robust to float jitter / duplicates)
def _p_to_index(p_target: float) -> int:
    if K == 0:
        return 0
    j = int(np.argmin(np.abs(p_grid - p_target)))
    return j

# ---------- Subsampling with REPLICATES (nested within each replicate across p_grid) ----------
index_rows = []

for rep in range(N_REPS):
    rng = _rng_for_rep(rep)
    print(f"\n[rep] {rep} | nested sampling over {K} depths (p_grid asc)")

    # Per-depth accumulators
    grouped_per_depth = [[] for _ in range(K)]                         # lists of (rows, cols, data)
    reads_kept_per_cell_list = [np.zeros(n_cells, dtype=np.int64) for _ in range(K)]
    umis_surv_per_cell_list  = [np.zeros(n_cells, dtype=np.int64) for _ in range(K)]
    kept_reads_total_list    = [0 for _ in range(K)]
    kept_umis_total_list     = [0 for _ in range(K)]

    # Stream all shards; update ALL depths in a nested manner
    for si, sh in enumerate(shards, 1):
        s0 = time.time()
        df  = _read_shard(sh)
        r   = df["umi_reads"].to_numpy()
        cel = df["cell_idx"].to_numpy()
        gen = df["gene_idx"].to_numpy()
        F = r.size

        if F == 0:
            print(f"  [shard {si:02d}/{len(shards)}] empty shard? skipping.")
            continue

        # Process in family micro-batches to bound memory
        for start in range(0, F, FAMILY_BATCH_SIZE):
            end = min(start + FAMILY_BATCH_SIZE, F)
            rb  = r[start:end]
            cb  = cel[start:end]
            gb  = gen[start:end]

            # Nested binomial ladder across p_grid:
            # k_j ~ Binom(r, p_j) with k_{j+1} = k_j + Binom(r - k_j, (p_{j+1}-p_j)/(1-p_j))
            k = None
            p_prev = 0.0

            for j, p in enumerate(p_grid):
                if j == 0:
                    k = rng.binomial(rb, p).astype(np.int32, copy=False)
                else:
                    remain = rb - k
                    denom = max(1e-12, 1.0 - p_prev)
                    q = (p - p_prev) / denom if p > p_prev else 0.0
                    q = float(np.clip(q, 0.0, 1.0))  # guard
                    if remain.dtype != np.int32:
                        remain = remain.astype(np.int32, copy=False)
                    k = k + rng.binomial(remain, q).astype(np.int32, copy=False)

                # per-cell kept reads
                if k.size:
                    pc_reads = (
                        pd.DataFrame({"cell_idx": cb, "kept": k}, dtype=np.int32)
                        .groupby("cell_idx", sort=False)["kept"].sum()
                        .reset_index()
                    )
                    reads_kept_per_cell_list[j][pc_reads["cell_idx"].to_numpy()] += pc_reads["kept"].to_numpy(np.int64)
                    kept_reads_total_list[j] += int(pc_reads["kept"].sum())

                # UMI survival at this depth (≥1 kept read)
                survive = (k > 0)
                if survive.any():
                    sel_cells = cb[survive]
                    sel_genes = gb[survive]

                    # each surviving family contributes 1 UMI to (cell, gene)
                    df_chunk = pd.DataFrame(
                        {"cell_idx": sel_cells, "gene_idx": sel_genes, "umis": 1},
                        dtype=np.int32
                    )
                    agg = df_chunk.groupby(["cell_idx","gene_idx"], sort=False)["umis"].sum().reset_index()

                    grouped_per_depth[j].append((
                        agg["cell_idx"].to_numpy(np.int32),
                        agg["gene_idx"].to_numpy(np.int32),
                        agg["umis"].to_numpy(np.int32),
                    ))

                    pc_umis = (
                        pd.DataFrame({"cell_idx": sel_cells, "one": 1}, dtype=np.int32)
                        .groupby("cell_idx", sort=False)["one"].sum()
                        .reset_index()
                    )
                    umis_surv_per_cell_list[j][pc_umis["cell_idx"].to_numpy()] += pc_umis["one"].to_numpy(np.int64)
                    kept_umis_total_list[j] += int(pc_umis["one"].sum())

                p_prev = p  # advance ladder

        dt = time.time() - s0
        last_reads = kept_reads_total_list[K-1] if K > 0 else 0
        print(f"  [shard {si:02d}/{len(shards)}] processed in {dt:.1f}s | cum_reads@max≈{last_reads:,}")

    # ----- After all shards for this replicate: build per-depth AnnData & write -----
    for t, p_target in zip(target_rpcs, p_for_target):
        j = _p_to_index(p_target)
        tag = f"rpc{t:d}"
        out_path = os.path.join(SAVE_DIR, f"subsample_MAPPED_{tag}_r{rep}.h5ad")
        if os.path.exists(out_path):
            print(f"[skip] exists → {out_path}")
            index_rows.append(dict(target_rpc=int(t), rep=int(rep), h5ad=out_path, p_keep=float(p_target)))
            continue

        # Build counts matrix
        X = _build_counts_matrix(n_cells, n_genes, grouped_per_depth[j])

        # Compose AnnData (reuse reference metadata to keep identity)
        ad = sc.AnnData(
            X=X,
            obs=ad_ref.obs.copy(),
            var=ad_ref.var.copy(),
            obsm=ad_ref.obsm.copy() if ad_ref.obsm is not None else None,
            uns=ad_ref.uns.copy() if ad_ref.uns is not None else None,
        )

        # Per-cell stats + saturation (10x definition: 1 - UMIs/reads)
        reads_kept = reads_kept_per_cell_list[j]
        umis_surv  = umis_surv_per_cell_list[j]
        ad.obs[f"reads_sim_MAPPED_{tag}"] = reads_kept
        ad.obs[f"umis_sim_MAPPED_{tag}"]  = umis_surv
        with np.errstate(divide="ignore", invalid="ignore"):
            sat = 1.0 - (umis_surv.astype(float) / np.clip(reads_kept, 1, None))
            sat[reads_kept == 0] = np.nan
        ad.obs[f"saturation_sim_MAPPED_{tag}"] = sat

        # Provenance
        ad.uns.setdefault("subsample_meta", {})
        ad.uns["subsample_meta"].update({
            "target_rpc": int(t),
            "rep": int(rep),
            "p_keep": float(p_target),
            "reads_in_shards": int(reads_in_shards),
            "n_cells": int(n_cells),
            "n_genes": int(n_genes),
            "seed_base": int(RNG_SEED_BASE),
            "nested_depths": [float(x) for x in p_grid.tolist()],
        })

        _ensure_lognorm(ad, LAYER_LOGNORM)
        ad.write_h5ad(out_path)
        print(f"[done] wrote {out_path} | kept_reads={kept_reads_total_list[j]:,} | kept_UMIs={kept_umis_total_list[j]:,}")

        index_rows.append(dict(target_rpc=int(t), rep=int(rep), h5ad=out_path, p_keep=float(p_target)))

# Save an index for discovery later (includes 'rep'); guard empty
index_csv = os.path.join(SAVE_DIR, "subsample_runs_index.csv")
if len(index_rows) == 0:
    print("[warn] No subsamples were created (index is empty).")
    # write an empty index with expected columns to avoid downstream KeyError
    pd.DataFrame(columns=["target_rpc","rep","h5ad","p_keep"]).to_csv(index_csv, index=False)
else:
    pd.DataFrame(index_rows).sort_values(["target_rpc","rep"]).to_csv(index_csv, index=False)
    print(f"\n[save] run index → {index_csv}")

print("[note] Nested sampling within each replicate ensures monotone depth curves and comparability across targets.")


[ref] 9,381 cells × 38,645 genes
[manifest] 1 shards
  [scan] shard 01/1 | +reads=10,670,749 | cum_reads=10,670,749
[totals] reads_in_shards=10,670,749 | families=1,368,465 | 0.0s
[feasible] max_rpc=1,137 (reads_in_shards 10,670,749 / cells 9,381)
[targets] [50, 170, 291, 412, 533, 653, 774, 895, 1016, 1137]

[rep] 0 | nested sampling over 10 depths (p_grid asc)
  [shard 01/1] processed in 4.2s | cum_reads@max≈10,666,033
normalizing counts per cell
    finished (0:00:00)
[done] wrote /homes/olymp/julius.guntrum/tmp/Koeln/bone/runs_rpc_clamped/subsample_MAPPED_rpc50_r0.h5ad | kept_reads=469,954 | kept_UMIs=383,877
normalizing counts per cell
    finished (0:00:00)
[done] wrote /homes/olymp/julius.guntrum/tmp/Koeln/bone/runs_rpc_clamped/subsample_MAPPED_rpc170_r0.h5ad | kept_reads=1,593,883 | kept_UMIs=874,501
normalizing counts per cell
    finished (0:00:00)
[done] wrote /homes/olymp/julius.guntrum/tmp/Koeln/bone/runs_rpc_clamped/subsample_MAPPED_rpc291_r0.h5ad | kept_reads=2,729,584 |

In [3]:
# ============================================
# STEP 3 — QC metrics + plots (replicate-aware) and artifact schema
# ============================================
import os
import json
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse

# -------- PATHS (re-use variables from previous chunks) --------
# Uses: OUT_DIR (workspace root from Task 1), REF_ADATA (deep run reference)
RUNS_DIR    = os.path.join(OUT_DIR, "runs_rpc_clamped")      # from STEP 2b
METRICS_DIR = os.path.join(OUT_DIR, "metrics_rpc_clamped")   # new
os.makedirs(METRICS_DIR, exist_ok=True)

index_csv = os.path.join(RUNS_DIR, "subsample_runs_index.csv")
assert os.path.exists(index_csv), f"Run index not found: {index_csv}"

In [8]:
# ============================================
# STEP 3 — QC metrics + plots (replicate-aware) and artifact schema
# ============================================
import os
import json
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse

# -------- PATHS (re-use variables from previous chunks) --------
# Uses: OUT_DIR (workspace root from Task 1), REF_ADATA (deep run reference)
RUNS_DIR    = os.path.join(OUT_DIR, "runs_rpc_clamped")      # from STEP 2b
METRICS_DIR = os.path.join(OUT_DIR, "metrics_rpc_clamped")   # new
os.makedirs(METRICS_DIR, exist_ok=True)

index_csv = os.path.join(RUNS_DIR, "subsample_runs_index.csv")
assert os.path.exists(index_csv), f"Run index not found: {index_csv}"

# -------- Load runs index, normalize dtypes --------
runs = pd.read_csv(index_csv)
if runs.empty:
    raise RuntimeError("Run index is empty — nothing to plot. Please run STEP 2b first.")

required_cols = {"h5ad", "p_keep", "rep", "target_rpc"}
missing = required_cols - set(runs.columns)
if missing:
    raise RuntimeError(f"Run index missing columns: {missing}")

runs["target_rpc"] = runs["target_rpc"].astype(int)
runs["rep"] = runs["rep"].astype(int)
runs = runs.sort_values(["target_rpc", "rep"]).reset_index(drop=True)

print("[index]")
print(runs)

# -------- Helpers --------
def _per_cell_genes(ad):
    X = ad.X
    if sparse.issparse(X):
        return np.asarray(X.getnnz(axis=1)).ravel()
    return (np.asarray(X) > 0).sum(axis=1)

def _metrics_for_run(h5ad_path, target_rpc, rep):
    ad = sc.read_h5ad(h5ad_path)
    tag = f"rpc{int(target_rpc)}"
    rcol = f"reads_sim_MAPPED_{tag}"
    ucol = f"umis_sim_MAPPED_{tag}"
    scol = f"saturation_sim_MAPPED_{tag}"

    for c in (rcol, ucol, scol):
        if c not in ad.obs.columns:
            raise RuntimeError(f"Missing expected column in {os.path.basename(h5ad_path)}: {c}")

    reads_per_cell = ad.obs[rcol].to_numpy(dtype=float)
    umis_per_cell  = ad.obs[ucol].to_numpy(dtype=float)
    genes_per_cell = _per_cell_genes(ad).astype(float)
    sat_per_cell   = ad.obs[scol].to_numpy(dtype=float)

    total_reads = float(np.nansum(reads_per_cell))
    total_umis  = float(np.nansum(umis_per_cell))

    out = dict(
        target_rpc=int(target_rpc),
        rep=int(rep),
        h5ad=h5ad_path,
        p_keep=float(ad.uns.get("subsample_meta", {}).get("p_keep", np.nan)),
        n_cells=int(ad.n_obs),
        total_reads=total_reads,
        total_umis=total_umis,
        median_reads=float(np.nanmedian(reads_per_cell)),
        mean_reads=float(np.nanmean(reads_per_cell)),
        median_umis=float(np.nanmedian(umis_per_cell)),
        mean_umis=float(np.nanmean(umis_per_cell)),
        median_genes=float(np.nanmedian(genes_per_cell)),
        mean_genes=float(np.nanmean(genes_per_cell)),
        median_saturation=float(np.nanmedian(sat_per_cell)),
        mean_saturation=float(np.nanmean(sat_per_cell)),
    )

    # lightweight consistency check (not fatal)
    try:
        umis_from_X = float(np.asarray(ad.X.sum(axis=1)).ravel().sum())
        if abs(umis_from_X - total_umis) > 0.01 * max(1.0, total_umis):
            print(f"[warn] UMIs mismatch (X vs obs) for {os.path.basename(h5ad_path)}: "
                  f"{umis_from_X:.0f} vs {total_umis:.0f}")
    except Exception as e:
        print(f"[note] skipped UMIs-from-X check: {e}")

    return out

def _agg_mean_sd(df, cols, xcol="reads_millions"):
    g = df.groupby("target_rpc", as_index=False)
    base = g[xcol].agg(x_mean="mean", x_sd="std")
    out = base.copy()
    for c in cols:
        agg = g[c].agg(**{f"{c}_mean": "mean", f"{c}_sd": "std"})
        out = out.merge(agg, on="target_rpc", how="left")
    out = out.merge(g.size().rename(columns={"size": "n"}), on="target_rpc", how="left")
    return out.sort_values("x_mean")

# -------- Compute per-rep metrics over all runs --------
rows = []
for _, r in runs.iterrows():
    print(f"[metrics] rpc={int(r['target_rpc'])} rep={int(r['rep'])}  {os.path.basename(r['h5ad'])}")
    rows.append(_metrics_for_run(r["h5ad"], r["target_rpc"], r["rep"]))

perrep_df = pd.DataFrame(rows).sort_values(["target_rpc","rep"]).reset_index(drop=True)
perrep_df["reads_millions"] = perrep_df["total_reads"] / 1e6

# -------- Aggregate across replicates (mean ± SD) --------
metric_cols = [
    "median_reads","median_umis","median_genes","median_saturation",
    "mean_reads","mean_umis","mean_genes","mean_saturation",
    "total_reads","n_cells"
]
agg_df = _agg_mean_sd(perrep_df, metric_cols, xcol="reads_millions")
agg_df["reads_millions_mean"] = agg_df["x_mean"]
agg_df["reads_millions_sd"]   = agg_df["x_sd"]
agg_df = agg_df.drop(columns=["x_mean","x_sd"])

# -------- Save CSVs (per-rep + aggregated) --------
perrep_csv = os.path.join(METRICS_DIR, "metrics_rpc_clamped_perrep.csv")
agg_csv    = os.path.join(METRICS_DIR, "metrics_rpc_clamped_agg.csv")
compat_csv = os.path.join(METRICS_DIR, "metrics_rpc_clamped.csv")  # alias for downstream code

perrep_df.to_csv(perrep_csv, index=False)
agg_df.to_csv(agg_csv, index=False)
agg_df.to_csv(compat_csv, index=False)

print(f"[save] per-rep metrics   → {perrep_csv}")
print(f"[save] aggregated metrics→ {agg_csv}")
print(f"[save] (compat alias)   → {compat_csv}")

# -------- Plotting (medians; per-rep scatter + mean±SD error bars) --------
plt.figure(figsize=(12, 9))
plots = [
    ("median_reads_mean",      "median_reads_sd",      "Median reads/cell"),
    ("median_umis_mean",       "median_umis_sd",       "Median UMIs/cell"),
    ("median_genes_mean",      "median_genes_sd",      "Median genes/cell"),
    ("median_saturation_mean", "median_saturation_sd", "Median saturation"),
]
for i, (col_mean, col_sd, label) in enumerate(plots, 1):
    ax = plt.subplot(2, 2, i)
    base_col = col_mean.replace("_mean", "")
    ax.scatter(perrep_df["reads_millions"], perrep_df[base_col], s=15, alpha=0.4)
    ax.errorbar(
        agg_df["reads_millions_mean"], agg_df[col_mean], yerr=agg_df[col_sd],
        fmt="-o", capsize=3
    )
    ax.set_xlabel("Total reads kept (millions)")
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
plt.suptitle("Subsampling (RPC-clamped): medians vs total kept reads (mean±SD across reps)", y=1.02)
plt.tight_layout()
panel_med_png = os.path.join(METRICS_DIR, "panel_medians_vs_total_reads.png")
plt.savefig(panel_med_png, dpi=180, bbox_inches="tight")
plt.close()
print(f"[save] {panel_med_png}")

# -------- Plotting (means; per-rep scatter + mean±SD error bars) --------
plt.figure(figsize=(12, 9))
plots_mean = [
    ("mean_reads_mean",      "mean_reads_sd",      "Mean reads/cell"),
    ("mean_umis_mean",       "mean_umis_sd",       "Mean UMIs/cell"),
    ("mean_genes_mean",      "mean_genes_sd",      "Mean genes/cell"),
    ("mean_saturation_mean", "mean_saturation_sd", "Mean saturation"),
]
for i, (col_mean, col_sd, label) in enumerate(plots_mean, 1):
    ax = plt.subplot(2, 2, i)
    base_col = col_mean.replace("_mean", "")
    ax.scatter(perrep_df["reads_millions"], perrep_df[base_col], s=15, alpha=0.4)
    ax.errorbar(
        agg_df["reads_millions_mean"], agg_df[col_mean], yerr=agg_df[col_sd],
        fmt="-o", capsize=3
    )
    ax.set_xlabel("Total reads kept (millions)")
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
plt.suptitle("Subsampling (RPC-clamped): means vs total kept reads (mean±SD across reps)", y=1.02)
plt.tight_layout()
panel_mean_png = os.path.join(METRICS_DIR, "panel_means_vs_total_reads.png")
plt.savefig(panel_mean_png, dpi=180, bbox_inches="tight")
plt.close()
print(f"[save] {panel_mean_png}")

# -------- Artifact schema to help future loading --------
schema = dict(
    paths=dict(
        runs_dir=RUNS_DIR,
        metrics_dir=METRICS_DIR,
        runs_index=index_csv,
        per_rep_metrics=perrep_csv,
        aggregated_metrics=agg_csv,
        aggregated_metrics_alias=compat_csv,
        figures=[panel_med_png, panel_mean_png],
    ),
    runs_index=dict(
        columns=list(runs.columns),
        description="Index of subsample runs (target_rpc, p_keep, h5ad path, rep).",
        reads_col_pattern="reads_sim_MAPPED_rpc{target_rpc}",
        umis_col_pattern="umis_sim_MAPPED_rpc{target_rpc}",
        saturation_col_pattern="saturation_sim_MAPPED_rpc{target_rpc}",
        rep_column="rep",
        has_replicates=True,
    ),
    metrics=dict(
        per_rep_columns=list(perrep_df.columns),
        aggregated_columns=list(agg_df.columns),
        x_axis_aggregated="reads_millions_mean",
        primary_metrics=[
            "median_reads_mean","median_umis_mean","median_genes_mean","median_saturation_mean",
            "mean_reads_mean","mean_umis_mean","mean_genes_mean","mean_saturation_mean",
        ],
        description="Per-replicate QC metrics and aggregated (mean±SD) across replicates.",
    ),
    reference=dict(
        ref_anndata=REF_ADATA,
        note="Reference (deep run) AnnData used to fix HVGs/PCA/UMAP in subsequent steps.",
    ),
)
schema_json = os.path.join(METRICS_DIR, "analysis_artifact_schema.json")
with open(schema_json, "w") as f:
    json.dump(schema, f, indent=2)
print(f"[save] schema → {schema_json}")

print("\n[done] Replicate-aware QC metrics computed and plots saved.")


[index]
    target_rpc  rep                                               h5ad  \
0           50    0  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
1           50    1  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
2           50    2  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
3          170    0  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
4          170    1  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
5          170    2  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
6          291    0  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
7          291    1  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
8          291    2  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
9          412    0  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
10         412    1  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
11         412    2  /homes/olymp/julius.guntrum/tmp/Koeln/bone/run...   
12         533    0  /homes/ol

In [5]:
# ============================================
# STEP 4–6 (novo) — Reference vs Cold-Start with QC
# ============================================
import os, json, math, warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import scanpy as sc
import umap  # umap-learn
import squidpy as sq
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, accuracy_score
from sklearn.neighbors import NearestNeighbors
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 2
OUT_DIR = "/homes/olymp/julius.guntrum/tmp/Koeln/skin_car"
# -------- PATHS (re-use variables from previous chunks) --------
RUNS_DIR       = os.path.join(OUT_DIR, "runs_rpc_clamped")
ANALYSIS_DIR   = os.path.join(OUT_DIR, "analysis_novo")   # <-- new destination
FIG_DIR        = os.path.join(ANALYSIS_DIR, "figs")
os.makedirs(ANALYSIS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

index_csv = os.path.join(RUNS_DIR, "subsample_runs_index.csv")
assert os.path.exists(index_csv), f"Run index not found: {index_csv}"

/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [10]:
ANALYSIS_MODE = "reference"  # "reference" or "coldstart"

# QC controls
APPLY_QC   = False
QC_MODE    = "absolute"     # "absolute" or "depth_scaled"
MIN_UMIS   = 1            # absolute baseline
MIN_GENES  = 10            # absolute baseline
MAX_MITO   = 0.15           # optional (if obs['pct_counts_mt'] present)
GENE_SCALE_EXP = 0.7        # genes ~ UMIs^alpha for depth scaling

# Embedding/graph params
N_HVG      = 2000
N_PCS      = 50
N_NEIGH    = 20
UMAP_MIND  = 0.5
RAND_STATE = 0

In [38]:
# ============================================
# STEP 4–6 (novo) — Reference vs Cold-Start with QC
# ============================================
import os, json, math, warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import scanpy as sc
import umap  # umap-learn
import squidpy as sq
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, accuracy_score
from sklearn.neighbors import NearestNeighbors
from scipy.optimize import linear_sum_assignment
REF_ADATA = ("/homes/olymp/julius.guntrum/tmp/Koeln/bone/bone_ref.h5ad")

warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 2
OUT_DIR = "/homes/olymp/julius.guntrum/tmp/Koeln/bone"
# -------- PATHS (re-use variables from previous chunks) --------
RUNS_DIR       = os.path.join(OUT_DIR, "runs_rpc_clamped")
ANALYSIS_DIR   = os.path.join(OUT_DIR, "analysis_novo_2")   # <-- new destination
FIG_DIR        = os.path.join(ANALYSIS_DIR, "figs")
os.makedirs(ANALYSIS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

index_csv = os.path.join(RUNS_DIR, "subsample_runs_index.csv")
assert os.path.exists(index_csv), f"Run index not found: {index_csv}"

# --- Tunables ---
# Analysis mode:
#   "reference" → ingest into reference PCA space (upper-bound comparability)
#   "coldstart" → de novo HVG/PCA/graph/Leiden per fraction (realistic cold start)
ANALYSIS_MODE = "coldstart"  # "reference" or "coldstart"

# QC controls
APPLY_QC   = False
QC_MODE    = "absolute"     # "absolute" or "depth_scaled"
MIN_UMIS   = 50            # absolute baseline
MIN_GENES  = 10            # absolute baseline
MAX_MITO   = 0.15           # optional (if obs['pct_counts_mt'] present)
GENE_SCALE_EXP = 0.7        # genes ~ UMIs^alpha for depth scaling

# Embedding/graph params
N_HVG      = 2000
N_PCS      = 50
N_NEIGH    = 20
UMAP_MIND  = 0.5
RAND_STATE = 0

def ensure_lognorm(ad: sc.AnnData, layer="lognorm"):
    if layer not in ad.layers:
        norm = sc.pp.normalize_total(ad, target_sum=1e4, inplace=False)
        ad.layers[layer] = norm["X"]
        sc.pp.log1p(ad, layer=layer)

# ---------- Reference model on the deepest data (used in reference-mode & for comparisons) ----------
REF_OUT_H5AD = os.path.join(ANALYSIS_DIR, "reference_with_embedding.h5ad")
PARAMS_JSON  = os.path.join(ANALYSIS_DIR, "reference_params.json")

def prepare_reference(ref_path: str) -> Tuple[sc.AnnData, umap.UMAP]:
    print(f"[ref] loading {ref_path}")
    ad_ref = sc.read_h5ad(ref_path)
    ensure_lognorm(ad_ref, "lognorm")

    print(f"[ref] selecting HVGs (seurat flavor; n_top={N_HVG})")
    sc.pp.highly_variable_genes(ad_ref, flavor="seurat", n_top_genes=N_HVG, layer="lognorm", inplace=True)
    hv_mask = ad_ref.var["highly_variable"].values

    print(f"[ref] PCA({N_PCS}) / neighbors({N_NEIGH})")
    sc.pp.pca(ad_ref, n_comps=N_PCS, use_highly_variable=True, layer="lognorm",
              svd_solver="arpack", random_state=RAND_STATE)
    sc.pp.neighbors(ad_ref, n_neighbors=N_NEIGH, n_pcs=N_PCS, use_rep="X_pca", random_state=RAND_STATE)

    print(f"[ref] fitting UMAP(min_dist={UMAP_MIND})")
    umap_model = umap.UMAP(min_dist=UMAP_MIND, random_state=RAND_STATE, n_neighbors=N_NEIGH, n_components=2)
    ad_ref.obsm["X_umap_ref"] = umap_model.fit_transform(ad_ref.obsm["X_pca"])

    print("[ref] Leiden clustering")
    sc.tl.leiden(ad_ref, resolution=0.6, flavor="igraph", key_added="leiden_ref", random_state=RAND_STATE)

    # (Optional) spatial neighbors + Moran's I on top expressed HVGs
    if "spatial" in ad_ref.obsm_keys():
        try:
            sq.gr.spatial_neighbors(ad_ref, coord_type="generic")
            hv_genes = ad_ref.var_names[hv_mask]
            means = np.asarray(ad_ref[:, hv_mask].layers["lognorm"].mean(axis=0)).ravel()
            top_idx = np.argsort(means)[::-1][:100]
            sq.gr.spatial_autocorr(ad_ref, mode="moran", genes=list(hv_genes[top_idx]))
        except Exception as e:
            print(f"[warn] spatial neighbors/Moran's I skipped: {e}")

    ad_ref.write_h5ad(REF_OUT_H5AD)
    with open(PARAMS_JSON, "w") as f:
        json.dump(dict(N_HVG=N_HVG, N_PCS=N_PCS, N_NEIGH=N_NEIGH, UMAP_MIND=UMAP_MIND, RAND_STATE=RAND_STATE), f, indent=2)
    print(f"[save] reference → {REF_OUT_H5AD}")
    return ad_ref, umap_model

ad_ref, umap_model = prepare_reference(REF_ADATA)


[ref] loading /homes/olymp/julius.guntrum/tmp/Koeln/bone/bone_ref.h5ad
normalizing counts per cell
    finished (0:00:00)
[ref] selecting HVGs (seurat flavor; n_top=2000)
extracting highly variable genes
    finished (0:00:00)
[ref] PCA(50) / neighbors(20)
computing PCA
    with n_comps=50


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


    finished (0:00:00)
computing neighbors
    finished (0:00:02)
[ref] fitting UMAP(min_dist=0.5)
[ref] Leiden clustering
running Leiden clustering
    finished (0:00:00)
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
Calculating moran's statistic for `None` permutations using `1` core(s)
Adding `adata.uns['moranI']`
Finish (0:00:00)
[save] reference → /homes/olymp/julius.guntrum/tmp/Koeln/bone/analysis_novo_2/reference_with_embedding.h5ad


In [39]:

from scipy.stats import spearmanr  # NEW: for DE log2FC Spearman


# ---------- QC & helpers ----------
def per_cell_umis_genes(ad: sc.AnnData):
    X = ad.X
    if sparse.issparse(X):
        umis = np.asarray(X.sum(axis=1)).ravel()
        genes = np.asarray((X > 0).sum(axis=1)).ravel()
    else:
        X = np.asarray(X)
        umis = X.sum(axis=1)
        genes = (X > 0).sum(axis=1)
    return umis, genes

def qc_mask(ad: sc.AnnData, p_keep: float | None, mode: str):
    umis, genes = per_cell_umis_genes(ad)
    if (mode != "depth_scaled") or (p_keep is None) or (not np.isfinite(p_keep)):
        min_umis = MIN_UMIS
        min_genes = MIN_GENES
    else:
        min_umis = max(int(np.floor(MIN_UMIS * p_keep)), 1)
        min_genes = max(int(np.floor(MIN_GENES * (p_keep ** GENE_SCALE_EXP))), 50)
    ok = (umis >= min_umis) & (genes >= min_genes)
    if "pct_counts_mt" in ad.obs:
        ok &= (np.asarray(ad.obs["pct_counts_mt"]).astype(float) <= MAX_MITO)
    return ok, dict(min_umis=min_umis, min_genes=min_genes)

def subset_graph(C: sparse.csr_matrix, mask: np.ndarray) -> sparse.csr_matrix:
    C = C.tocsr()
    idx = np.where(mask)[0]
    return C[idx][:, idx].tocsr()

# ---------- optional spatial (reuse the reference coordinates if identical cells) ----------
def add_spatial_graphs_and_svgs(ad: sc.AnnData, max_genes_moran: int = 50, hv_mask=None):
    if "spatial" not in ad.obsm_keys():
        if "spatial" in ad_ref.obsm_keys() and ad.n_obs == ad_ref.n_obs and np.all(ad.obs_names == ad_ref.obs_names):
            ad.obsm["spatial"] = ad_ref.obsm["spatial"].copy()
        else:
            return
    try:
        sq.gr.spatial_neighbors(ad, coord_type="generic")
        if hv_mask is None:
            genes = ad.var_names
        else:
            genes = ad.var_names[hv_mask]
        means = np.asarray((ad[:, genes].layers["lognorm"] if "lognorm" in ad.layers else ad[:, genes].X).mean(axis=0)).ravel()
        order = np.argsort(means)[::-1][:max_genes_moran]
        sq.gr.spatial_autocorr(ad, mode="moran", genes=list(pd.Index(genes)[order]))
    except Exception as e:
        print(f"[warn] spatial neighbors/Moran's I failed: {e}")

# ---------- metrics ----------
def hungarian_weighted_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, Dict[str, str]]:
    classes_true = pd.Categorical(y_true).categories
    classes_pred = pd.Categorical(y_pred).categories
    ct = pd.crosstab(pd.Categorical(y_true, categories=classes_true),
                     pd.Categorical(y_pred, categories=classes_pred)).values
    if ct.size == 0:
        return np.nan, {}
    row_ind, col_ind = linear_sum_assignment(ct.max() - ct)
    acc = ct[row_ind, col_ind].sum() / ct.sum() if ct.sum() > 0 else np.nan
    mapping = {str(classes_true[i]): str(classes_pred[j]) for i, j in zip(row_ind, col_ind)}
    return acc, mapping

def topk_jaccard_from_connectivities(C_ref: sparse.csr_matrix, C_run: sparse.csr_matrix, k: int = 15) -> float:
    assert C_ref.shape == C_run.shape
    n = C_ref.shape[0]
    A = C_ref.tocsr(); B = C_run.tocsr()
    def topk_indices_csr(mat: sparse.csr_matrix, kk: int) -> List[np.ndarray]:
        indptr, indices, data = mat.indptr, mat.indices, mat.data
        out = []
        for i in range(n):
            start, end = indptr[i], indptr[i+1]
            if end - start <= kk:
                idx = indices[start:end]
            else:
                row_idx = indices[start:end]
                row_dat = data[start:end]
                topk = np.argpartition(row_dat, -(kk))[-kk:]
                idx = row_idx[topk]
            out.append(idx[idx != i])
        return out
    topA = topk_indices_csr(A, k)
    topB = topk_indices_csr(B, k)
    jac = []
    for i in range(n):
        a = set(topA[i]); b = set(topB[i])
        if not a and not b:
            jac.append(1.0)
        else:
            inter = len(a & b); union = len(a | b)
            jac.append(inter / union if union > 0 else 0.0)
    return float(np.mean(jac))

# ---------- DE helpers (NEW) ----------
def _deg_table(ad: sc.AnnData, key: str, group: str) -> pd.DataFrame:
    """Robustly pull an RG result table."""
    try:
        return sc.get.rank_genes_groups_df(ad, key=key, group=group)
    except Exception:
        rg = ad.uns[key]
        names = pd.Series(rg["names"][group], name="names")
        pvals_adj = pd.Series(rg["pvals_adj"][group], name="pvals_adj")
        logfc = pd.Series(rg["logfoldchanges"][group], name="logfoldchanges")
        return pd.DataFrame({"names": names, "pvals_adj": pvals_adj, "logfoldchanges": logfc})

def _topN_jaccard(list_a, list_b, N=20):
    if len(list_a)==0 and len(list_b)==0: return np.nan
    a = set(list_a[:N]); b = set(list_b[:N])
    u = len(a|b)
    return len(a&b)/u if u else np.nan

def _ensure_de(ad: sc.AnnData, groupby: str, key_added: str, n_genes: int = 300):
    """Compute rank_genes_groups if missing under key_added on this object."""
    if key_added not in ad.uns:
        sc.tl.rank_genes_groups(ad, groupby=groupby, method="wilcoxon",
                                use_raw=False, layer="lognorm",
                                key_added=key_added, n_genes=n_genes)

def _compute_de_metrics(
    ad_ref_sub: sc.AnnData,
    ad_run_sub: sc.AnnData,
    ref_group_key: str,
    run_group_key: str,
    mapping_true_to_pred: dict[str, str],
    topN: int = 20
) -> tuple[float, float]:
    """
    Returns (DE Top-20 Jaccard mean across matched clusters, DE log2FC Spearman mean).
    mapping_true_to_pred is from Hungarian match on (ref labels → run labels).
    """
    # Compute DE tables (on the *same* cells that were used for metrics)
    _ensure_de(ad_ref_sub, groupby=ref_group_key, key_added="__de_ref_tmp")
    _ensure_de(ad_run_sub, groupby=run_group_key, key_added="__de_run_tmp")

    # invert mapping: run_label -> ref_label
    mapping_pred_to_true = {v: k for k, v in mapping_true_to_pred.items()}

    jcats, rhos = [], []
    for qlab, tlab in mapping_pred_to_true.items():
        # pull DE tables
        try:
            df_ref = _deg_table(ad_ref_sub, "__de_ref_tmp", str(tlab)).sort_values("pvals_adj")
            df_run = _deg_table(ad_run_sub, "__de_run_tmp", str(qlab)).sort_values("pvals_adj")
        except Exception:
            continue

        # Top-20 Jaccard by adjusted p-value
        jcats.append(_topN_jaccard(df_ref["names"].tolist(), df_run["names"].tolist(), N=topN))

        # Spearman on log2FC over overlapping genes
        m = df_ref[["names","logfoldchanges"]].merge(
            df_run[["names","logfoldchanges"]], on="names",
            suffixes=("_ref","_run")
        )
        if len(m) >= 10:
            rhos.append(spearmanr(m["logfoldchanges_ref"], m["logfoldchanges_run"], nan_policy="omit").correlation)

    de_top20 = float(np.nanmean(jcats)) if len(jcats) else np.nan
    de_rho   = float(np.nanmean(rhos))  if len(rhos)  else np.nan
    return de_top20, de_rho


# ---------- Mode 1: reference/ingest + QC (metrics computed on QC-retained cells) ----------
def run_reference_mode_with_qc(h5ad_path: str, t_rpc: int, p_keep: float | None) -> Tuple[Dict, str]:
    ad = sc.read_h5ad(h5ad_path)
    ensure_lognorm(ad, "lognorm")

    # Ingest PCA only
    Xb = ad.X
    try:
        ad.X = ad.layers["lognorm"].astype(np.float32, copy=False)
        sc.tl.ingest(ad, ad_ref, obs=None, embedding_method=["pca"])
    finally:
        ad.X = Xb
    sc.pp.neighbors(ad, n_neighbors=N_NEIGH, n_pcs=N_PCS, use_rep="X_pca", random_state=RAND_STATE)
    #ad.obsm["X_umap_refspace"] = umap_model.transform(ad.obsm["X_pca"])
    sc.tl.leiden(ad, resolution=0.6, flavor="igraph", key_added="leiden_ds", random_state=RAND_STATE)

    # QC on original object (compute once)
    mask, qc_thr = qc_mask(ad, p_keep, QC_MODE) if APPLY_QC else (np.ones(ad.n_obs, dtype=bool), dict(min_umis=None, min_genes=None))
    cells_kept = int(mask.sum()); frac_kept = float(cells_kept / ad.n_obs)
    umis, genes = per_cell_umis_genes(ad)
    med_genes_kept = float(np.median(genes[mask])) if cells_kept > 0 else np.nan

    # metrics on retained set
    ref_sub = ad_ref[ad.obs_names[mask]].copy()
    y_ref = ref_sub.obs["leiden_ref"].to_numpy()
    y_ds  = ad.obs["leiden_ds"].to_numpy()[mask]

    ari = adjusted_rand_score(y_ref, y_ds)
    nmi = normalized_mutual_info_score(y_ref, y_ds)
    hung_acc, mapping_true_to_pred = hungarian_weighted_accuracy(y_ref, y_ds)
    mapping_true_to_pred = mapping_true_to_pred or {}
    jac = topk_jaccard_from_connectivities(subset_graph(ad_ref.obsp["connectivities"], mask), subset_graph(ad.obsp["connectivities"], mask), k=15)

    # NEW: DE metrics (compute on QC-retained cells)
    ad_run_sub = ad[mask].copy()
    de_top20, de_rho = _compute_de_metrics(ref_sub, ad_run_sub, "leiden_ref", "leiden_ds", mapping_true_to_pred)

    # Label-transfer acc (KNN in PCA space) restricted to retained cells
    nn = NearestNeighbors(n_neighbors=N_NEIGH)
    nn.fit(ad_ref.obsm["X_pca"])
    neigh_idx = nn.kneighbors(ad.obsm["X_pca"][mask], return_distance=False)
    ref_labels = ad_ref.obs["leiden_ref"].to_numpy()
    pred = np.array([pd.Series(ref_labels[row]).value_counts().index[0] for row in neigh_idx], dtype=object)
    acc_transfer = accuracy_score(y_ref, pred)

    # Totals based on run's recorded reads column
    tag = f"rpc{int(t_rpc)}"
    reads_all = float(np.nansum(ad.obs.get(f"reads_sim_MAPPED_{tag}", 0)))
    reads_kept = float(np.nansum(np.asarray(ad.obs.get(f"reads_sim_MAPPED_{tag}", 0))[mask]))

    # Save projected object (keep everything; QC mask saved in obs)
    ad.obs[f"qc_keep_{QC_MODE}"] = mask
    out_proj = os.path.join(ANALYSIS_DIR, f"proj_{os.path.basename(h5ad_path)}")
    ad.write_h5ad(out_proj)

    metrics = dict(
        ARI=ari, NMI=nmi, hungarian_acc=hung_acc, knn_jaccard=jac,
        de_top20_jaccard=de_top20, de_log2fc_spearman=de_rho,  # NEW
        label_transfer_acc=acc_transfer,
        total_reads=reads_all, total_reads_kept=reads_kept,
        cells_total=int(ad.n_obs), cells_kept=cells_kept, frac_kept=frac_kept,
        median_genes_kept=med_genes_kept,
        qc_min_umis=qc_thr["min_umis"], qc_min_genes=qc_thr["min_genes"]
    )
    return metrics, out_proj

# ---------- Mode 2: cold-start (de novo) + QC ----------
def run_coldstart_with_qc(h5ad_path: str, t_rpc: int, p_keep: float | None) -> Tuple[Dict, str]:
    ad = sc.read_h5ad(h5ad_path)
    ensure_lognorm(ad, "lognorm")

    # QC & subset before HVG/PCA (realistic)
    mask, qc_thr = qc_mask(ad, p_keep, QC_MODE) if APPLY_QC else (np.ones(ad.n_obs, dtype=bool), dict(min_umis=None, min_genes=None))
    ad_q = ad[mask].copy()
    cells_kept = int(ad_q.n_obs); frac_kept = float(cells_kept / ad.n_obs)
    umis_full, genes_full = per_cell_umis_genes(ad)
    med_genes_kept = float(np.median(genes_full[mask])) if cells_kept > 0 else np.nan

    # de novo HVG / PCA / neighbors / UMAP / Leiden on QC-retained set
    sc.pp.highly_variable_genes(ad_q, flavor="seurat", n_top_genes=N_HVG, layer="lognorm", inplace=True)
    hv_mask = ad_q.var["highly_variable"].values
    sc.pp.pca(ad_q, n_comps=N_PCS, use_highly_variable=True, layer="lognorm",
              svd_solver="arpack", random_state=RAND_STATE)
    sc.pp.neighbors(ad_q, n_neighbors=N_NEIGH, n_pcs=N_PCS, use_rep="X_pca", random_state=RAND_STATE)
    #sc.tl.umap(ad_q, min_dist=UMAP_MIND, random_state=RAND_STATE)
    sc.tl.leiden(ad_q, resolution=0.6, flavor="igraph", key_added="leiden_novo", random_state=RAND_STATE)
    add_spatial_graphs_and_svgs(ad_q, max_genes_moran=50, hv_mask=hv_mask)

    # metrics vs reference on retained cells
    ref_sub = ad_ref[ad_q.obs_names].copy()
    y_ref = ref_sub.obs["leiden_ref"].to_numpy()
    y_ds  = ad_q.obs["leiden_novo"].to_numpy()

    ari = adjusted_rand_score(y_ref, y_ds)
    nmi = normalized_mutual_info_score(y_ref, y_ds)
    hung_acc, mapping_true_to_pred = hungarian_weighted_accuracy(y_ref, y_ds)
    mapping_true_to_pred = mapping_true_to_pred or {}

    jac = topk_jaccard_from_connectivities(subset_graph(ad_ref.obsp["connectivities"], ad.obs_names.isin(ad_q.obs_names)),
                                           ad_q.obsp["connectivities"], k=15)
    de_top20, de_rho = _compute_de_metrics(ref_sub, ad_q, "leiden_ref", "leiden_novo", mapping_true_to_pred)

    # Totals
    tag = f"rpc{int(t_rpc)}"
    reads_all  = float(np.nansum(ad.obs.get(f"reads_sim_MAPPED_{tag}", 0)))
    reads_kept = float(np.nansum(np.asarray(ad.obs.get(f"reads_sim_MAPPED_{tag}", 0))[mask]))

    # Save result (QC-retained, de novo object)
    out_proj = os.path.join(ANALYSIS_DIR, f"novo_{os.path.basename(h5ad_path)}")
    ad_q.write_h5ad(out_proj)

    metrics = dict(
        ARI=ari, NMI=nmi, hungarian_acc=hung_acc, knn_jaccard=jac,
        de_top20_jaccard=de_top20, de_log2fc_spearman=de_rho,  # NEW
        label_transfer_acc=np.nan,  # not meaningful in cold-start
        total_reads=reads_all, total_reads_kept=reads_kept,
        cells_total=int(ad.n_obs), cells_kept=cells_kept, frac_kept=frac_kept,
        median_genes_kept=med_genes_kept,
        qc_min_umis=qc_thr["min_umis"], qc_min_genes=qc_thr["min_genes"]
    )
    return metrics, out_proj


In [40]:
# -------- Load runs index (with repetitions) --------
runs = pd.read_csv(index_csv).sort_values(["target_rpc","rep"]).reset_index(drop=True)
assert {"target_rpc","rep","h5ad"}.issubset(runs.columns), "Runs index must contain target_rpc, rep, h5ad"
if "p_keep" not in runs.columns:
    runs["p_keep"] = np.nan  # depth-scaled QC will fall back to absolute if missing

# -------- Per-run analysis --------
metrics_rows = []
for _, r in runs.iterrows():
    h5ad = r["h5ad"]
    t_rpc = int(r["target_rpc"]); rep = int(r["rep"])
    p_keep = float(r["p_keep"]) if pd.notna(r["p_keep"]) else None
    print(f"[{ANALYSIS_MODE}+QC] rpc={t_rpc} rep={rep}  {os.path.basename(h5ad)}  p_keep={p_keep}")

    if ANALYSIS_MODE == "reference":
        met, out_path = run_reference_mode_with_qc(h5ad, t_rpc, p_keep)
    elif ANALYSIS_MODE == "coldstart":
        met, out_path = run_coldstart_with_qc(h5ad, t_rpc, p_keep)
    else:
        raise ValueError("ANALYSIS_MODE must be 'reference' or 'coldstart'.")

    metrics_rows.append(dict(target_rpc=t_rpc, rep=rep, h5ad=out_path, **met))

metrics_df = pd.DataFrame(metrics_rows).sort_values(["target_rpc","rep"]).reset_index(drop=True)
metrics_df["reads_millions"] = metrics_df["total_reads"] / 1e6
metrics_df["reads_millions_kept"] = metrics_df["total_reads_kept"] / 1e6

metrics_csv = os.path.join(ANALYSIS_DIR, "clustering_stability_metrics.csv")
metrics_df.to_csv(metrics_csv, index=False)
print(f"[save] stability+QC metrics → {metrics_csv}")

# -------- Aggregate (mean ± SD across reps) --------
agg = metrics_df.groupby("target_rpc").agg(
    reads_millions_mean=("reads_millions","mean"),
    reads_millions_sd=("reads_millions","std"),
    ARI_mean=("ARI","mean"), ARI_sd=("ARI","std"),
    NMI_mean=("NMI","mean"), NMI_sd=("NMI","std"),
    hungarian_acc_mean=("hungarian_acc","mean"), hungarian_acc_sd=("hungarian_acc","std"),
    knn_jaccard_mean=("knn_jaccard","mean"), knn_jaccard_sd=("knn_jaccard","std"),
    de_top20_jaccard_mean=("de_top20_jaccard","mean"), de_top20_jaccard_sd=("de_top20_jaccard","std"),   # NEW
    de_log2fc_spearman_mean=("de_log2fc_spearman","mean"), de_log2fc_spearman_sd=("de_log2fc_spearman","std"),  # NEW
    label_transfer_acc_mean=("label_transfer_acc","mean"), label_transfer_acc_sd=("label_transfer_acc","std"),
    cells_kept_mean=("cells_kept","mean"), cells_kept_sd=("cells_kept","std"),
    frac_kept_mean=("frac_kept","mean"), frac_kept_sd=("frac_kept","std"),
    median_genes_kept_mean=("median_genes_kept","mean"), median_genes_kept_sd=("median_genes_kept","std"),
    n=("ARI","size")
).reset_index()


# -------- Plot helpers --------
def _panel(base_df, agg_df, base_col, mean_col, sd_col, ylab, fname, xcol="reads_millions", xlab="Total reads kept (millions)"):
    plt.figure(figsize=(7.5, 5.5))
    plt.scatter(base_df[xcol], base_df[base_col], s=18, alpha=0.45)
    plt.errorbar(agg_df[f"{xcol}_mean"] if f"{xcol}_mean" in agg_df.columns else agg_df[xcol],
                 agg_df[mean_col], yerr=agg_df[sd_col], fmt="-o", capsize=3)
    plt.xlabel(xlab); plt.ylabel(ylab); plt.grid(True, alpha=0.3)
    outp = os.path.join(FIG_DIR, fname)
    plt.tight_layout(); plt.savefig(outp, dpi=180, bbox_inches="tight"); plt.close()
    print(f"[save] {outp}")

# -------- Stability plots (mean ± SD) --------
_panel(metrics_df, agg, "ARI", "ARI_mean", "ARI_sd", "ARI (vs ref)", f"{ANALYSIS_MODE}_stability_ARI.png")
_panel(metrics_df, agg, "NMI", "NMI_mean", "NMI_sd", "NMI (vs ref)", f"{ANALYSIS_MODE}_stability_NMI.png")
_panel(metrics_df, agg, "hungarian_acc", "hungarian_acc_mean", "hungarian_acc_sd",
       "Hungarian accuracy (weighted)", f"{ANALYSIS_MODE}_stability_HungarianAcc.png")
_panel(metrics_df, agg, "knn_jaccard", "knn_jaccard_mean", "knn_jaccard_sd",
       "kNN Jaccard overlap (K=15)", f"{ANALYSIS_MODE}_stability_kNNJaccard.png")

# -------- Combined panel (NEW): stability + DE (and label-transfer if available) --------
def _big_panel(base_df, agg_df, out_name, xcol="reads_millions"):
    # Build metric spec dynamically (skip metrics that are all-NaN)
    specs = [
        ("ARI", "ARI_mean", "ARI_sd", "ARI (vs ref)"),
        ("NMI", "NMI_mean", "NMI_sd", "NMI (vs ref)"),
        ("hungarian_acc", "hungarian_acc_mean", "hungarian_acc_sd", "Hungarian accuracy (weighted)"),
        ("knn_jaccard", "knn_jaccard_mean", "knn_jaccard_sd", "kNN Jaccard overlap (K=15)"),
        ("de_top20_jaccard", "de_top20_jaccard_mean", "de_top20_jaccard_sd", "DE Top-20 Jaccard"),
        ("de_log2fc_spearman", "de_log2fc_spearman_mean", "de_log2fc_spearman_sd", "DE log2FC Spearman"),
    ]
    # include label transfer if present
    if (ANALYSIS_MODE == "reference") and ("label_transfer_acc" in base_df.columns) and (not base_df["label_transfer_acc"].isna().all()):
        specs.insert(3, ("label_transfer_acc", "label_transfer_acc_mean", "label_transfer_acc_sd", "Label-transfer accuracy (KNN in PCA)"))

    # filter out any metrics that are completely missing
    specs = [s for s in specs if s[0] in base_df.columns]

    n_plots = len(specs)
    ncols = 3
    nrows = int(math.ceil(n_plots / ncols))
    plt.figure(figsize=(4.2*ncols, 3.8*nrows))
    for i, (base_col, mean_col, sd_col, title) in enumerate(specs, start=1):
        ax = plt.subplot(nrows, ncols, i)
        ax.scatter(base_df[xcol], base_df[base_col], s=14, alpha=0.4)
        x_mean_col = f"{xcol}_mean" if f"{xcol}_mean" in agg_df.columns else xcol
        ax.errorbar(agg_df[x_mean_col], agg_df[mean_col], yerr=agg_df[sd_col], fmt="-o", capsize=3)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Total reads (millions)")
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    outp = os.path.join(FIG_DIR, out_name)
    plt.savefig(outp, dpi=180, bbox_inches="tight"); plt.close()
    print(f"[save] combined metrics panel → {outp}")

_big_panel(metrics_df, agg, out_name=f"{ANALYSIS_MODE}_combined_metrics_panel.png")


# Label transfer accuracy only meaningful in reference-mode
if ANALYSIS_MODE == "reference" and not metrics_df["label_transfer_acc"].isna().all():
    _panel(metrics_df, agg, "label_transfer_acc", "label_transfer_acc_mean", "label_transfer_acc_sd",
           "Label-transfer accuracy (KNN in PCA)", f"{ANALYSIS_MODE}_stability_LabelTransferAcc.png")

# -------- QC plots (cells & genes retained) --------
_panel(metrics_df, agg, "cells_kept", "cells_kept_mean", "cells_kept_sd",
       "# Cells retained (QC)", f"{ANALYSIS_MODE}_qc_cells_retained.png")
_panel(metrics_df, agg, "median_genes_kept", "median_genes_kept_mean", "median_genes_kept_sd",
       "Median genes/cell (QC-retained)", f"{ANALYSIS_MODE}_qc_median_genes_kept.png")

# # -------- UMAP grid across depths (choose the first rep per RPC) --------
# sel = metrics_df.sort_values(["target_rpc","rep"]).groupby("target_rpc").head(1)
# n_pan = sel.shape[0]
# ncols = min(4, n_pan)
# nrows = int(math.ceil(n_pan / ncols))
# plt.figure(figsize=(4*ncols, 4.2*nrows))
# for i, row in enumerate(sel.itertuples(), start=1):
#     ad = sc.read_h5ad(row.h5ad)
#     ax = plt.subplot(nrows, ncols, i)
#     if ANALYSIS_MODE == "reference":
#         basis = "X_umap_refspace"; color = "leiden_ds" if "leiden_ds" in ad.obs else None
#     else:
#         basis = "X_umap" if "X_umap" in ad.obsm_keys() else "X_umap_refspace"
#         color = "leiden_novo" if "leiden_novo" in ad.obs else ("leiden_ds" if "leiden_ds" in ad.obs else None)
#     sc.pl.embedding(ad, basis=basis, color=color, ax=ax, show=False, title=f"rpc{row.target_rpc}")
# plt.tight_layout()
# umap_grid_png = os.path.join(FIG_DIR, f"{ANALYSIS_MODE}_umap_grid.png")
# plt.savefig(umap_grid_png, dpi=180, bbox_inches="tight"); plt.close()
# print(f"[save] {umap_grid_png}")

# -------- Update analysis schema --------
schema = dict(
    mode=ANALYSIS_MODE,
    reference=dict(
        ref_h5ad=REF_OUT_H5AD,
        params=json.load(open(PARAMS_JSON, "r")),
        note="Reference is fitted once on deep run. 'reference' mode ingests PCA; 'coldstart' recomputes de novo."
    ),
    runs_index=index_csv,
    stability_metrics_csv=metrics_csv,
    qc=dict(
        apply_qc=APPLY_QC, qc_mode=QC_MODE, min_umis=MIN_UMIS, min_genes=MIN_GENES, max_mito=MAX_MITO, gene_scale_exp=GENE_SCALE_EXP
    ),
    figures={f: os.path.join(FIG_DIR, f) for f in os.listdir(FIG_DIR)}
)
schema_json = os.path.join(ANALYSIS_DIR, "analysis_artifact_schema.json")
with open(schema_json, "w") as f:
    json.dump(schema, f, indent=2)
print(f"[save] schema → {schema_json}")

print(f"\n[done:{ANALYSIS_MODE}] QC applied, metrics aggregated with error bars, QC metrics plotted, figures in {FIG_DIR}")


[coldstart+QC] rpc=50 rep=0  subsample_MAPPED_rpc50_r0.h5ad  p_keep=0.0439566144794521
extracting highly variable genes
    finished (0:00:00)
computing PCA
    with n_comps=50


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


    finished (0:00:00)
computing neighbors
    finished (0:00:02)
running Leiden clustering
    finished (0:00:00)
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
Calculating moran's statistic for `None` permutations using `1` core(s)
Adding `adata.uns['moranI']`
Finish (0:00:00)
ranking genes
    finished (0:00:13)
ranking genes
    finished (0:00:13)


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:456: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:458: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/s

[coldstart+QC] rpc=50 rep=1  subsample_MAPPED_rpc50_r1.h5ad  p_keep=0.0439566144794521
extracting highly variable genes
    finished (0:00:00)
computing PCA
    with n_comps=50


...

    finished (0:00:00)
computing neighbors
    finished (0:00:02)
running Leiden clustering
    finished (0:00:00)
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
Calculating moran's statistic for `None` permutations using `1` core(s)
Adding `adata.uns['moranI']`
Finish (0:00:00)
ranking genes
    finished (0:00:13)
ranking genes
    finished (0:00:13)
[coldstart+QC] rpc=1137 rep=1  subsample_MAPPED_rpc1137_r1.h5ad  p_keep=0.9995734132627428
extracting highly variable genes
    finished (0:00:00)
computing PCA
    with n_comps=50


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


    finished (0:00:00)
computing neighbors
    finished (0:00:02)
running Leiden clustering
    finished (0:00:00)
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
Calculating moran's statistic for `None` permutations using `1` core(s)
Adding `adata.uns['moranI']`
Finish (0:00:00)
ranking genes
    finished (0:00:13)
ranking genes
    finished (0:00:13)
[coldstart+QC] rpc=1137 rep=2  subsample_MAPPED_rpc1137_r2.h5ad  p_keep=0.9995734132627428
extracting highly variable genes
    finished (0:00:00)
computing PCA
    with n_comps=50


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


    finished (0:00:00)
computing neighbors
    finished (0:00:02)
running Leiden clustering
    finished (0:00:00)
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
Calculating moran's statistic for `None` permutations using `1` core(s)
Adding `adata.uns['moranI']`
Finish (0:00:00)
ranking genes
    finished (0:00:13)
ranking genes
    finished (0:00:13)
[save] stability+QC metrics → /homes/olymp/julius.guntrum/tmp/Koeln/bone/analysis_novo_2/clustering_stability_metrics.csv
[save] /homes/olymp/julius.guntrum/tmp/Koeln/bone/analysis_novo_2/figs/coldstart_stability_ARI.png
[save] /homes/olymp/julius.guntrum/tmp/Koeln/bone/analysis_novo_2/figs/coldstart_stability_NMI.png
[save] /homes/olymp/julius.guntrum/tmp/Koeln/bone/analysis_novo_2/figs/coldstart_stability_HungarianAcc.png
[save] /homes/olymp/julius.guntrum/tmp

In [8]:
# ============================================
# De-novo depth analysis with robust QC guards
# - QC per fraction (no stable cohort)
# - De-novo HVG/PCA/graph (+ optional UMAP)
# - Viability checks to avoid HVG bin-edge NaNs
# - Subset by reps and/or fraction stride
# - Error-bar plots + QC plots
# Outputs: analysis_novo/
# ============================================
import os, json, math, warnings
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 2

# ---------------- CONFIG ----------------
OUT_ROOT   = OUT_DIR
RUNS_DIR   = os.path.join(OUT_ROOT, "runs_rpc_clamped")
INDEX_CSV  = os.path.join(RUNS_DIR, "subsample_runs_index.csv")

# Deep reference for baseline labels (if missing, pick max target_rpc from index)
REF_DEEP_H5AD = os.path.join(OUT_ROOT, "analysis_novo", "reference_with_embedding.h5ad")

# New output directory
ANALYSIS_DIR = os.path.join(OUT_ROOT, "analysis_novo")
FIG_DIR      = os.path.join(ANALYSIS_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)

# Tunables
N_HVG         = 3000
N_PCS         = 50
N_NEIGH       = 20
LEIDEN_RES    = 0.6
RAND_STATE    = 0
KNN_OVERLAP_K = 15

# QC thresholds (absolute)
MIN_GENES = 0
MIN_UMIS  = 10

# Viability thresholds (guards before HVG/PCA)
MIN_CELLS_FOR_MODEL       = 50          # require at least this many cells post-QC
MIN_NONZERO_GENES_GLOBAL  = 200         # across all cells in the run
MIN_TOTAL_UMIS_GLOBAL     = 10_000      # total UMIs across QC-kept cells
MIN_PCS                   = 2           # minimum PCs to compute

# Optional UMAP compute + plots
DO_UMAP = False  # set True to compute/plot UMAPs per fraction

# Subset speed-ups
REPS_PER_FRACTION   = 3  # e.g., 1 or 2; None = use all reps available
FRACTION_STRIDE     = 1     # 1=use all; 2=every other; 3=every third, etc.

# ---------------- Helpers ----------------
def ensure_lognorm(ad: sc.AnnData, layer="lognorm"):
    if layer not in ad.layers:
        norm = sc.pp.normalize_total(ad, target_sum=1e4, inplace=False)
        ad.layers[layer] = norm["X"]
        sc.pp.log1p(ad, layer=layer)

def per_cell_genes(ad: sc.AnnData):
    X = ad.X
    if sparse.issparse(X):
        return np.asarray(X.getnnz(axis=1)).ravel()
    return (np.asarray(X) > 0).sum(axis=1)

def qc_mask(ad: sc.AnnData, min_genes=MIN_GENES, min_umis=MIN_UMIS):
    X = ad.X
    if sparse.issparse(X):
        umis  = np.asarray(X.sum(axis=1)).ravel()
        genes = np.asarray((X > 0).sum(axis=1)).ravel()
    else:
        Xd = np.asarray(X)
        umis  = Xd.sum(axis=1).ravel()
        genes = (Xd > 0).sum(axis=1).ravel()
    return (genes >= min_genes) & (umis >= min_umis)

def _total_umis(ad: sc.AnnData) -> float:
    X = ad.X
    if sparse.issparse(X):
        return float(np.asarray(X.sum(axis=1)).ravel().sum())
    return float(np.asarray(X).sum())

def _nonzero_genes_global(ad: sc.AnnData) -> int:
    X = ad.X
    if sparse.issparse(X):
        return int((np.asarray(X.sum(axis=0)).ravel() > 0).sum())
    return int((np.asarray(X).sum(axis=0).ravel() > 0).sum())

def viability_check(ad: sc.AnnData) -> Tuple[bool, str]:
    """Return (is_viable, reason_if_not)."""
    if ad.n_obs < MIN_CELLS_FOR_MODEL:
        return False, f"cells<{MIN_CELLS_FOR_MODEL}"
    nzg = _nonzero_genes_global(ad)
    if nzg < MIN_NONZERO_GENES_GLOBAL:
        return False, f"nonzero_genes<{MIN_NONZERO_GENES_GLOBAL} (got {nzg})"
    tot = _total_umis(ad)
    if tot < MIN_TOTAL_UMIS_GLOBAL:
        return False, f"total_UMIs<{MIN_TOTAL_UMIS_GLOBAL} (got {int(tot)})"
    return True, ""

def fit_hvg_pca_graph(ad: sc.AnnData, n_hvg=N_HVG, n_pcs=N_PCS, n_neigh=N_NEIGH) -> int:
    """Fit HVG/PCA/Neighbors; returns effective n_pcs used. Raises on internal errors."""
    sc.pp.highly_variable_genes(ad, flavor="seurat", n_top_genes=n_hvg, layer="lognorm", inplace=True)
    n_hv = int(ad.var["highly_variable"].sum())
    # choose an effective number of PCs the data can support
    n_pcs_eff = max(MIN_PCS, min(n_pcs, ad.n_obs - 1, n_hv - 1 if n_hv > 1 else MIN_PCS))
    if n_pcs_eff < MIN_PCS:
        raise RuntimeError(f"Too few PCs supportable (hv={n_hv}, cells={ad.n_obs})")
    sc.pp.pca(ad, n_comps=n_pcs_eff, use_highly_variable=True, layer="lognorm",
              svd_solver="arpack", random_state=RAND_STATE)
    sc.pp.neighbors(ad, n_neighbors=n_neigh, n_pcs=n_pcs_eff, use_rep="X_pca", random_state=RAND_STATE)
    return n_pcs_eff

def leiden(ad: sc.AnnData, res=LEIDEN_RES):
    sc.tl.leiden(ad, resolution=res, flavor="igraph", key_added="leiden_ds", random_state=RAND_STATE)

def knn_adj_from_pca(X_pca: np.ndarray, k=KNN_OVERLAP_K):
    nn = NearestNeighbors(n_neighbors=k+1, algorithm="auto", metric="euclidean").fit(X_pca)
    ind = nn.kneighbors(return_distance=False)
    return ind[:,1:]  # drop self

def knn_jaccard_index(knn_a: np.ndarray, knn_b: np.ndarray) -> float:
    out = []
    for a, b in zip(knn_a, knn_b):
        sa, sb = set(a), set(b)
        u = len(sa | sb)
        out.append(len(sa & sb)/u if u else 1.0)
    return float(np.mean(out)) if len(out) else np.nan

def hungarian_match_acc(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, Dict[str, str]]:
    tcat = pd.Categorical(y_true)
    pcat = pd.Categorical(y_pred)
    cm = np.zeros((len(tcat.categories), len(pcat.categories)), dtype=int)
    for t, p in zip(tcat.codes, pcat.codes):
        if t >= 0 and p >= 0:
            cm[t, p] += 1
    if cm.sum() == 0:
        return np.nan, {}
    r, c = linear_sum_assignment(cm.max() - cm)
    acc = cm[r, c].sum() / cm.sum()
    mapping = {str(pcat.categories[cj]): str(tcat.categories[ri]) for ri, cj in zip(r, c)}
    return float(acc), mapping

def total_reads_from_obs(ad: sc.AnnData, target_rpc: int) -> float:
    tag = f"rpc{int(target_rpc)}"
    col = f"reads_sim_MAPPED_{tag}"
    if col in ad.obs:
        return float(np.asarray(ad.obs[col]).sum())
    # fallback: sum of counts
    X = ad.X
    return float(np.asarray(X.sum(axis=1)).ravel().sum())

# ---------------- Load index & reference ----------------
assert os.path.exists(INDEX_CSV), f"Run index not found: {INDEX_CSV}"
runs = pd.read_csv(INDEX_CSV)
if "rep" not in runs.columns:
    runs["rep"] = 0
runs["target_rpc"] = runs["target_rpc"].astype(int)
runs["rep"] = runs["rep"].astype(int)

# stride fractions
rpcs_sorted   = sorted(runs["target_rpc"].unique().tolist())
rpcs_selected = rpcs_sorted[::int(FRACTION_STRIDE)]
runs = runs[runs["target_rpc"].isin(rpcs_selected)].copy()

# limit reps per fraction
runs = runs.sort_values(["target_rpc", "rep"]).reset_index(drop=True)
if REPS_PER_FRACTION is not None:
    runs = runs.groupby("target_rpc", as_index=False).head(int(REPS_PER_FRACTION)).reset_index(drop=True)

print("[index/selected]")
print(runs)

# reference
if os.path.exists(REF_DEEP_H5AD):
    ad_ref = sc.read_h5ad(REF_DEEP_H5AD)
else:
    full_runs = pd.read_csv(INDEX_CSV)
    full_runs["target_rpc"] = full_runs["target_rpc"].astype(int)
    row = full_runs.iloc[full_runs["target_rpc"].astype(int).argmax()]
    ad_ref = sc.read_h5ad(row["h5ad"])
    print(f"[ref] using {os.path.basename(row['h5ad'])} as deep reference")
ensure_lognorm(ad_ref, "lognorm")
if "leiden_ref" not in ad_ref.obs.columns:
    sc.pp.highly_variable_genes(ad_ref, flavor="seurat", n_top_genes=N_HVG, layer="lognorm", inplace=True)
    sc.pp.pca(ad_ref, n_comps=min(N_PCS, max(MIN_PCS, ad_ref.n_obs-1)), use_highly_variable=True,
              layer="lognorm", svd_solver="arpack", random_state=RAND_STATE)
    sc.pp.neighbors(ad_ref, n_neighbors=N_NEIGH, n_pcs=min(N_PCS, max(MIN_PCS, ad_ref.n_obs-1)),
                    use_rep="X_pca", random_state=RAND_STATE)
    sc.tl.leiden(ad_ref, resolution=LEIDEN_RES, flavor="igraph", key_added="leiden_ref", random_state=RAND_STATE)
if "X_pca" not in ad_ref.obsm_keys():
    sc.pp.pca(ad_ref, n_comps=min(N_PCS, max(MIN_PCS, ad_ref.n_obs-1)), use_highly_variable=False,
              layer="lognorm", svd_solver="arpack", random_state=RAND_STATE)

# ---------------- Per-run processing ----------------
metrics_rows = []
qc_rows = []

for _, rr in runs.iterrows():
    rpc   = int(rr["target_rpc"])
    rep   = int(rr["rep"])
    h5ad  = str(rr["h5ad"])
    if not os.path.exists(h5ad):
        print(f"[skip] missing file: {h5ad}")
        continue

    print(f"\n[run] rpc={rpc} rep={rep} → {os.path.basename(h5ad)}")
    ad = sc.read_h5ad(h5ad)
    ensure_lognorm(ad, "lognorm")

    # QC per fraction
    mask = qc_mask(ad, MIN_GENES, MIN_UMIS)
    kept = ad[mask].copy()
    print(f"[QC] kept {kept.n_obs:,}/{ad.n_obs:,} cells "
          f"(min_genes={MIN_GENES}, min_umis={MIN_UMIS})")

    # record QC metrics (always)
    total_reads = total_reads_from_obs(ad, rpc)
    reads_millions = total_reads / 1e6
    median_genes_cell = float(np.median(per_cell_genes(kept))) if kept.n_obs else np.nan
    total_genes_detected = _nonzero_genes_global(kept) if kept.n_obs else 0

    qc_rows.append(dict(
        target_rpc=rpc, rep=rep,
        reads_millions=reads_millions,
        n_cells_kept=int(kept.n_obs),
        median_genes_per_cell=median_genes_cell,
        total_genes_detected=int(total_genes_detected)
    ))

    # viability gate before HVG/PCA
    viable, reason = viability_check(kept)
    if not viable:
        print(f"[skip-model] rpc={rpc} rep={rep} not viable for HVG/PCA → {reason}")
        out_h5ad = os.path.join(ANALYSIS_DIR, f"novo_rpc{rpc}_rep{rep}.h5ad")
        kept.write_h5ad(out_h5ad)
        metrics_rows.append(dict(
            target_rpc=rpc, rep=rep, h5ad=out_h5ad,
            total_reads=total_reads, reads_millions=reads_millions,
            ARI=np.nan, NMI=np.nan, hungarian_acc=np.nan, knn_jaccard=np.nan
        ))
        continue

    # de-novo HVG/PCA/graph (+ optional UMAP), then Leiden
    try:
        n_pcs_eff = fit_hvg_pca_graph(kept, n_hvg=N_HVG, n_pcs=N_PCS, n_neigh=N_NEIGH)
    except Exception as e:
        print(f"[skip-model] rpc={rpc} rep={rep} HVG/PCA failed: {e}")
        out_h5ad = os.path.join(ANALYSIS_DIR, f"novo_rpc{rpc}_rep{rep}.h5ad")
        kept.write_h5ad(out_h5ad)
        metrics_rows.append(dict(
            target_rpc=rpc, rep=rep, h5ad=out_h5ad,
            total_reads=total_reads, reads_millions=reads_millions,
            ARI=np.nan, NMI=np.nan, hungarian_acc=np.nan, knn_jaccard=np.nan
        ))
        continue

    if DO_UMAP:
        try:
            sc.tl.umap(kept, min_dist=0.5, random_state=RAND_STATE)
        except Exception as e:
            print(f"[warn] UMAP failed: {e}")

    leiden(kept, res=LEIDEN_RES)

    # Metrics vs reference on intersecting cells
    common = ad_ref.obs_names.intersection(kept.obs_names)
    if common.size == 0:
        print("[warn] no overlapping cell IDs with reference; metrics set to NaN.")
        ARI = NMI = hung_acc = knnJ = np.nan
    else:
        ref_sub = ad_ref[common].copy()
        run_sub = kept[common].copy()
        y_true  = ref_sub.obs["leiden_ref"].astype(str).values
        y_pred  = run_sub.obs["leiden_ds"].astype(str).values
        ARI = adjusted_rand_score(y_true, y_pred)
        NMI = normalized_mutual_info_score(y_true, y_pred)
        # Hungarian accuracy
        hung_acc, _ = hungarian_match_acc(y_true, y_pred)
        # kNN-Jaccard on PCA spaces (requires enough cells > K)
        if run_sub.n_obs <= KNN_OVERLAP_K or ref_sub.n_obs <= KNN_OVERLAP_K:
            knnJ = np.nan
        else:
            Xp_ref = np.asarray(ref_sub.obsm["X_pca"])
            Xp_run = np.asarray(run_sub.obsm["X_pca"])
            knn_r  = knn_adj_from_pca(Xp_ref, k=KNN_OVERLAP_K)
            knn_q  = knn_adj_from_pca(Xp_run, k=KNN_OVERLAP_K)
            knnJ   = knn_jaccard_index(knn_r, knn_q)

    # Save processed run
    out_h5ad = os.path.join(ANALYSIS_DIR, f"novo_rpc{rpc}_rep{rep}.h5ad")
    kept.write_h5ad(out_h5ad)

    metrics_rows.append(dict(
        target_rpc=rpc, rep=rep, h5ad=out_h5ad,
        total_reads=total_reads, reads_millions=reads_millions,
        ARI=float(ARI) if np.isfinite(ARI) else np.nan,
        NMI=float(NMI) if np.isfinite(NMI) else np.nan,
        hungarian_acc=float(hung_acc) if np.isfinite(hung_acc) else np.nan,
        knn_jaccard=float(knnJ) if np.isfinite(knnJ) else np.nan
    ))

    # Optional per-run UMAP plot
    if DO_UMAP and "X_umap" in kept.obsm_keys():
        plt.figure(figsize=(5.2, 4.6))
        sc.pl.umap(kept, color="leiden_ds", show=False, title=f"rpc{rpc} rep{rep}")
        fpath = os.path.join(FIG_DIR, f"umap_rpc{rpc}_rep{rep}.png")
        plt.savefig(fpath, dpi=160, bbox_inches="tight"); plt.close()
        print(f"[save] {fpath}")

# ---------------- Save metrics ----------------
metrics_df = pd.DataFrame(metrics_rows).sort_values(["target_rpc","rep"]).reset_index(drop=True)
qc_df      = pd.DataFrame(qc_rows).sort_values(["target_rpc","rep"]).reset_index(drop=True)

metrics_csv = os.path.join(ANALYSIS_DIR, "clustering_metrics_novo.csv")
qc_csv      = os.path.join(ANALYSIS_DIR, "qc_metrics_novo.csv")
metrics_df.to_csv(metrics_csv, index=False)
qc_df.to_csv(qc_csv, index=False)
print(f"[save] metrics → {metrics_csv}")
print(f"[save] QC     → {qc_csv}")

# ---------------- Aggregate & plot with error bars ----------------
def agg_with_err(df, ycols, xcol="reads_millions", gcol="target_rpc"):
    ag = df.groupby(gcol).agg(**{
        f"{xcol}_mean": (xcol, "mean"),
        f"{xcol}_sd":   (xcol, "std"),
        **{f"{c}_mean": (c, "mean") for c in ycols},
        **{f"{c}_sd":   (c, "std")  for c in ycols},
        "n":            (xcol, "size")
    }).reset_index()
    return ag

def plot_err(base_df, ag_df, base_y, mean_y, sd_y, ylabel, fname):
    # drop NaNs for scatter
    base = base_df[[ "reads_millions", base_y ]].dropna()
    plt.figure(figsize=(7.5, 5.5))
    if not base.empty:
        plt.scatter(base["reads_millions"], base[base_y], s=18, alpha=0.5)
    # errorbar: only where mean is finite
    agp = ag_df[[ "reads_millions_mean", mean_y, sd_y ]].dropna()
    if not agp.empty:
        plt.errorbar(agp["reads_millions_mean"], agp[mean_y],
                     yerr=agp[sd_y], fmt="-o", capsize=3)
    plt.xlabel("Total reads kept (millions)"); plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3); plt.tight_layout()
    outp = os.path.join(FIG_DIR, fname)
    plt.savefig(outp, dpi=180, bbox_inches="tight"); plt.close()
    print(f"[save] {outp}")

# Clustering metrics panel
if not metrics_df.empty:
    m_ag = agg_with_err(metrics_df, ["ARI","NMI","hungarian_acc","knn_jaccard"])
    plot_err(metrics_df, m_ag, "ARI",           "ARI_mean",           "ARI_sd",           "ARI",                      "stability_ARI_novo.png")
    plot_err(metrics_df, m_ag, "NMI",           "NMI_mean",           "NMI_sd",           "NMI",                      "stability_NMI_novo.png")
    plot_err(metrics_df, m_ag, "hungarian_acc", "hungarian_acc_mean", "hungarian_acc_sd", "Hungarian accuracy",       "stability_HungarianAcc_novo.png")
    plot_err(metrics_df, m_ag, "knn_jaccard",   "knn_jaccard_mean",   "knn_jaccard_sd",   f"kNN Jaccard (K={KNN_OVERLAP_K})", "stability_kNNJaccard_novo.png")

# QC metrics panel
if not qc_df.empty:
    q_ag = agg_with_err(qc_df, ["n_cells_kept","median_genes_per_cell","total_genes_detected"])
    plot_err(qc_df, q_ag, "n_cells_kept",          "n_cells_kept_mean",          "n_cells_kept_sd",          "# cells kept",            "qc_cells_kept_novo.png")
    plot_err(qc_df, q_ag, "median_genes_per_cell", "median_genes_per_cell_mean", "median_genes_per_cell_sd", "Median genes / cell",     "qc_median_genes_per_cell_novo.png")
    plot_err(qc_df, q_ag, "total_genes_detected",  "total_genes_detected_mean",  "total_genes_detected_sd",  "Total genes detected",    "qc_total_genes_detected_novo.png")

# Optional: UMAP grid (one rep per fraction)
if DO_UMAP and not metrics_df.empty:
    sel = metrics_df.sort_values(["target_rpc","rep"]).groupby("target_rpc").head(1)
    n_pan = sel.shape[0]
    if n_pan > 0:
        ncols = min(4, n_pan)
        nrows = int(math.ceil(n_pan / ncols))
        plt.figure(figsize=(4*ncols, 4.2*nrows))
        for i, row in enumerate(sel.itertuples(), start=1):
            adp = sc.read_h5ad(row.h5ad)
            ax = plt.subplot(nrows, ncols, i)
            if "X_umap" in adp.obsm_keys():
                sc.pl.umap(adp, color="leiden_ds", ax=ax, show=False, title=f"rpc{row.target_rpc}")
            else:
                ax.text(0.5, 0.5, "No UMAP", ha="center", va="center")
                ax.set_axis_off()
        plt.tight_layout()
        umap_grid_png = os.path.join(FIG_DIR, "umap_grid_leiden_ds_novo.png")
        plt.savefig(umap_grid_png, dpi=180, bbox_inches="tight"); plt.close()
        print(f"[save] {umap_grid_png}")

# ---------------- Persist small schema ----------------
schema = dict(
    inputs=dict(
        runs_index=INDEX_CSV,
        reference_h5ad=REF_DEEP_H5AD if os.path.exists(REF_DEEP_H5AD) else "auto(max target_rpc)",
    ),
    params=dict(
        n_hvg=N_HVG, n_pcs=N_PCS, n_neighbors=N_NEIGH, leiden_res=LEIDEN_RES,
        knn_overlap_k=KNN_OVERLAP_K, random_state=RAND_STATE,
        qc_min_genes=MIN_GENES, qc_min_umis=MIN_UMIS,
        viability=dict(
            min_cells=MIN_CELLS_FOR_MODEL,
            min_nonzero_genes=MIN_NONZERO_GENES_GLOBAL,
            min_total_umis=MIN_TOTAL_UMIS_GLOBAL
        ),
        do_umap=bool(DO_UMAP),
        reps_per_fraction=REPS_PER_FRACTION, fraction_stride=FRACTION_STRIDE
    ),
    outputs=dict(
        analysis_dir=ANALYSIS_DIR,
        clustering_metrics_csv=metrics_csv,
        qc_metrics_csv=qc_csv,
        figures=dict(
            ARI=os.path.join(FIG_DIR, "stability_ARI_novo.png"),
            NMI=os.path.join(FIG_DIR, "stability_NMI_novo.png"),
            HungarianAcc=os.path.join(FIG_DIR, "stability_HungarianAcc_novo.png"),
            kNNJaccard=os.path.join(FIG_DIR, "stability_kNNJaccard_novo.png"),
            qc_cells=os.path.join(FIG_DIR, "qc_cells_kept_novo.png"),
            qc_medgenes=os.path.join(FIG_DIR, "qc_median_genes_per_cell_novo.png"),
            qc_totalgenes=os.path.join(FIG_DIR, "qc_total_genes_detected_novo.png"),
        ),
    ),
)
with open(os.path.join(ANALYSIS_DIR, "analysis_novo_schema.json"), "w") as f:
    json.dump(schema, f, indent=2)

print("\n[done] De-novo analysis complete (robust QC guards, optional UMAP, subset controls, error-bar plots).")


[index/selected]
    target_rpc  rep                                               h5ad  \
0           50    0  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
1           50    1  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
2           50    2  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
3          217    0  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
4          217    1  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
5          217    2  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
6          385    0  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
7          385    1  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
8          385    2  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
9          552    0  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
10         552    1  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
11         552    2  /homes/olymp/julius.guntrum/tmp/Koeln/skin_car...   
12         720    0  

/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


    finished (0:00:01)
computing neighbors
    finished (0:00:37)
running Leiden clustering
    finished (0:00:04)

[run] rpc=50 rep=0 → subsample_MAPPED_rpc50_r0.h5ad
[QC] kept 46,412/51,713 cells (min_genes=0, min_umis=10)
extracting highly variable genes
    finished (0:00:00)
computing PCA
    with n_comps=50


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


    finished (0:00:00)
computing neighbors
    finished (0:00:07)
running Leiden clustering
    finished (0:00:02)

[run] rpc=50 rep=1 → subsample_MAPPED_rpc50_r1.h5ad
[QC] kept 46,372/51,713 cells (min_genes=0, min_umis=10)
extracting highly variable genes
    finished (0:00:00)
computing PCA
    with n_comps=50


...

    finished (0:00:01)
computing neighbors
    finished (0:00:08)
running Leiden clustering
    finished (0:00:02)

[run] rpc=1558 rep=1 → subsample_MAPPED_rpc1558_r1.h5ad
[QC] kept 51,216/51,713 cells (min_genes=0, min_umis=10)
extracting highly variable genes
    finished (0:00:00)
computing PCA
    with n_comps=50


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


    finished (0:00:01)
computing neighbors
    finished (0:00:08)
running Leiden clustering
    finished (0:00:02)

[run] rpc=1558 rep=2 → subsample_MAPPED_rpc1558_r2.h5ad
[QC] kept 51,216/51,713 cells (min_genes=0, min_umis=10)
extracting highly variable genes
    finished (0:00:00)
computing PCA
    with n_comps=50


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


    finished (0:00:01)
computing neighbors
    finished (0:00:08)
running Leiden clustering
    finished (0:00:02)
[save] metrics → /homes/olymp/julius.guntrum/tmp/Koeln/skin_car/analysis_novo/clustering_metrics_novo.csv
[save] QC     → /homes/olymp/julius.guntrum/tmp/Koeln/skin_car/analysis_novo/qc_metrics_novo.csv
[save] /homes/olymp/julius.guntrum/tmp/Koeln/skin_car/analysis_novo/figs/stability_ARI_novo.png
[save] /homes/olymp/julius.guntrum/tmp/Koeln/skin_car/analysis_novo/figs/stability_NMI_novo.png
[save] /homes/olymp/julius.guntrum/tmp/Koeln/skin_car/analysis_novo/figs/stability_HungarianAcc_novo.png
[save] /homes/olymp/julius.guntrum/tmp/Koeln/skin_car/analysis_novo/figs/stability_kNNJaccard_novo.png
[save] /homes/olymp/julius.guntrum/tmp/Koeln/skin_car/analysis_novo/figs/qc_cells_kept_novo.png
[save] /homes/olymp/julius.guntrum/tmp/Koeln/skin_car/analysis_novo/figs/qc_median_genes_per_cell_novo.png
[save] /homes/olymp/julius.guntrum/tmp/Koeln/skin_car/analysis_novo/figs/qc_tot

In [41]:
# ============================================
# STEP 6c — Marker trajectories (mean counts/cell vs depth)
# One plot with all markers; reuse existing projections/metrics.
# No % change vs ref; we plot absolute mean counts per cell.
# ============================================
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# ---------- Paths from previous steps ----------
RUNS_DIR     = os.path.join(OUT_DIR, "runs_rpc_clamped")
ANALYSIS_DIR = os.path.join(OUT_DIR, "analysis_rpc_clamped")
FIG_DIR      = os.path.join(ANALYSIS_DIR, "figs_3")
os.makedirs(FIG_DIR, exist_ok=True)

metrics_csv = os.path.join(ANALYSIS_DIR, "clustering_stability_metrics.csv")
assert os.path.exists(metrics_csv), f"Missing metrics CSV: {metrics_csv}"
metrics_df = pd.read_csv(metrics_csv).sort_values(["target_rpc","rep"]).reset_index(drop=True)

# Try to reuse per-rep extra metrics if available
extra_perrep_csv = os.path.join(ANALYSIS_DIR, "extra_metrics_perrep.csv")

MARKER_GENES = ["CD34","CD3E","CD4","CD68","CD8A","COL5A2","EPCAM","MS4A1","MZB1"]

def _get_gene_index(ad: sc.AnnData, gene_symbol: str):
    """Robustly map a gene symbol to var index if possible."""
    if gene_symbol in ad.var_names:
        return int(ad.var_names.get_loc(gene_symbol))
    for col in ["gene_symbols", "gene_symbol", "feature_name", "gene_name", "name", "symbols"]:
        if col in ad.var.columns.values:
            hits = np.where(ad.var[col].astype(str).values == gene_symbol)[0]
            if hits.size > 0:
                return int(hits[0])
    return None

def _mean_counts_per_cell(ad: sc.AnnData, gene_idx: int) -> float:
    """Mean UMI counts per cell for a single gene index (from ad.X)."""
    if gene_idx is None:
        return np.nan
    col = ad.X[:, gene_idx]
    try:
        s = float(col.sum())
    except Exception:
        s = float(np.asarray(col).sum())
    return s / max(1, ad.n_obs)

# ---------- Build (or load) per-replicate marker means ----------
need_compute = True
if os.path.exists(extra_perrep_csv):
    try:
        extra_df = pd.read_csv(extra_perrep_csv)
        have_cols = set(extra_df.columns)
        needed = {"target_rpc","rep","reads_millions"} | {f"{g}_mean" for g in MARKER_GENES}
        if needed.issubset(have_cols):
            marker_perrep = extra_df.loc[:, ["target_rpc","rep","reads_millions", *[f"{g}_mean" for g in MARKER_GENES]]].copy()
            need_compute = False
    except Exception:
        need_compute = True

if need_compute:
    rows = []
    for r in metrics_df.itertuples():
        ad = sc.read_h5ad(r.h5ad)
        vals = {}
        for g in MARKER_GENES:
            gi = _get_gene_index(ad, g)
            vals[f"{g}_mean"] = _mean_counts_per_cell(ad, gi)
        rows.append(dict(target_rpc=int(r.target_rpc), rep=int(r.rep),
                         reads_millions=float(r.total_reads)/1e6, **vals))
    marker_perrep = pd.DataFrame(rows).sort_values(["target_rpc","rep"]).reset_index(drop=True)

# ---------- Aggregate across replicates by depth ----------
agg_specs = {"reads_millions": ["mean", "std"]}
for g in MARKER_GENES:
    agg_specs[f"{g}_mean"] = ["mean", "std"]

agg_df = (marker_perrep
          .groupby("target_rpc")
          .agg(agg_specs))
agg_df.columns = ["_".join(col).strip() for col in agg_df.columns.values]
agg_df = agg_df.reset_index()
# Sort x-axis by actual mean reads
agg_df = agg_df.sort_values("reads_millions_mean").reset_index(drop=True)

# Save aggregated CSV for reuse
markers_agg_csv = os.path.join(ANALYSIS_DIR, "markers_mean_counts_agg.csv")
agg_df.to_csv(markers_agg_csv, index=False)
print(f"[save] markers (aggregated mean counts/cell) → {markers_agg_csv}")

# ---------- Plot single figure: all marker trajectories ----------
plt.figure(figsize=(10.5, 7.5))
x = agg_df["reads_millions_mean"].to_numpy()

for g in MARKER_GENES:
    y = agg_df[f"{g}_mean_mean"].to_numpy()
    ysd = agg_df[f"{g}_mean_std"].to_numpy() if f"{g}_mean_std" in agg_df.columns else None

    # line with markers
    plt.plot(x, y, "-o", label=g, linewidth=2, markersize=4, alpha=0.9)
    # optional ribbon for SD (only if available and not NaN)
    if ysd is not None and np.isfinite(ysd).any():
        y1 = y - ysd
        y2 = y + ysd
        plt.fill_between(x, y1, y2, alpha=0.12)

plt.xlabel("Total reads kept (millions)")
plt.ylabel("Mean counts per cell")
plt.title("Marker trajectories across depth (mean counts/cell)")
plt.grid(True, alpha=0.3)
plt.legend(ncol=3, frameon=False)
out_png = os.path.join(FIG_DIR, "marker_trajectories_mean_counts.png")
plt.tight_layout()
plt.savefig(out_png, dpi=200, bbox_inches="tight")
plt.close()
print(f"[save] {out_png}")

# ---------- Update schema ----------
schema_json = os.path.join(ANALYSIS_DIR, "analysis_artifact_schema.json")
schema = {}
if os.path.exists(schema_json):
    try:
        import json
        with open(schema_json, "r") as f:
            schema = json.load(f)
    except Exception:
        schema = {}
schema.setdefault("figures", {})
schema["figures"]["MarkerTrajectoriesMeanCounts"] = out_png
schema.setdefault("tables", {})
schema["tables"]["MarkerMeansAggCSV"] = markers_agg_csv
try:
    with open(schema_json, "w") as f:
        json.dump(schema, f, indent=2)
    print(f"[save] schema (updated) → {schema_json}")
except Exception as e:
    print(f"[warn] could not update schema: {e}")


[save] markers (aggregated mean counts/cell) → /homes/olymp/julius.guntrum/tmp/Koeln/bone/analysis_rpc_clamped/markers_mean_counts_agg.csv
[save] /homes/olymp/julius.guntrum/tmp/Koeln/bone/analysis_rpc_clamped/figs_3/marker_trajectories_mean_counts.png
[save] schema (updated) → /homes/olymp/julius.guntrum/tmp/Koeln/bone/analysis_rpc_clamped/analysis_artifact_schema.json


In [ ]:
duo_data= sc.read_h5ad("/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/runs_rpc_clamped/subsample_MAPPED_rpc930_r0.h5ad")

In [ ]:
duo_data.obs